1

In [15]:
import numpy as np
import pandas as pd
from collections import Counter
import lightgbm as lgb
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import ndcg_score
import difflib

train_app = pd.read_csv('applications_train.csv')
test_app = pd.read_csv('applications_test.csv')
jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')

train_df = train_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
test_df = test_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
eng_map = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
edu_map = {'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
exchange_rates = {'USD': 1.0, 'EUR': 1.08, 'IRR': 0.00002}

for df in [train_df, test_df]:
    df['english_score'] = df['english_proficiency'].map(eng_map).fillna(0)
    df['edu_score'] = df['education_level'].map(edu_map).fillna(0)
    
    df['application_date'] = pd.to_datetime(df['application_date'], dayfirst=True, format='mixed')
    df['job_posted_date'] = pd.to_datetime(df['job_posted_date'], dayfirst=True, format='mixed')
    df['days_to_apply'] = (df['application_date'] - df['job_posted_date']).dt.days.fillna(0)
    
    rates = df['salary_currency'].map(exchange_rates).fillna(1.0)
    df['salary_min_usd'] = df['salary_min'] * rates
    df['salary_max_usd'] = df['salary_max'] * rates
    
    df['expected_salary'] = df['expected_salary'].fillna(df['expected_salary'].median())
    df['salary_min_usd'] = df['salary_min_usd'].fillna(df['salary_min_usd'].median())
    df['salary_max_usd'] = df['salary_max_usd'].fillna(df['salary_max_usd'].median())
    
    df['salary_gap'] = df['expected_salary'] - ((df['salary_min_usd'] + df['salary_max_usd']) / 2)
    df['experience_gap'] = df['years_experience'] - df['min_years_experience']
    
    df['skill_match_ratio'] = df.apply(
        lambda row: len(set(str(row['skills']).split('|')) & set(str(row['required_skills']).split('|'))) / 
                    max(1, len(set(str(row['required_skills']).split('|')))), axis=1
    )
    df['title_similarity'] = df.apply(
        lambda row: difflib.SequenceMatcher(None, str(row['job_title']).lower(), str(row['current_title']).lower()).ratio(), axis=1
    )

2

In [16]:
def tokenize_pipe(s):
    if pd.isna(s):
        return set()
    return {t.strip().lower() for t in str(s).split("|") if t.strip()}

def tokenize_title(s):
    if pd.isna(s):
        return set()
    return {w for w in str(s).lower().replace(".", " ").split() if len(w) > 1}
def build_advanced_features(df, fitted_idf=None):
    cand_sk = df["skills"].apply(tokenize_pipe)
    req_sk = df["required_skills"].apply(tokenize_pipe)
    certs = df["certifications"].apply(tokenize_pipe)
    job_tok = df["job_title"].apply(tokenize_title)
    cur_tok = df["current_title"].apply(tokenize_title)

    df["skill_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(cand_sk, req_sk)]
    df["missing_required_skills"] = [len(b - a) for a, b in zip(cand_sk, req_sk)]
    df["extra_skills"] = [len(a - b) for a, b in zip(cand_sk, req_sk)]
    df["cert_skill_overlap"] = [len(c & b) for c, b in zip(certs, req_sk)]
    df["cert_title_overlap"] = [len(c & j) for c, j in zip(certs, job_tok)]
    df["n_certifications"] = certs.apply(len)
    df["title_token_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(job_tok, cur_tok)]

    # مدیریت TF-IDF برای جلوگیری از Data Shift
    if fitted_idf is None:
        dfreq = Counter()
        for s in req_sk:
            dfreq.update(s)
        n_docs = len(req_sk)
        idf = {k: np.log((1 + n_docs) / (1 + v)) + 1 for k, v in dfreq.items()}
    else:
        idf = fitted_idf

    vals = []
    for a, b in zip(cand_sk, req_sk):
        if not b:
            vals.append(0.0)
            continue
        num = sum(idf.get(t, 0.0) for t in (a & b))
        den = sum(idf.get(t, 0.0) for t in b)
        vals.append(num / den if den else 0.0)
    df["skill_match_idf"] = vals

    exp = df["years_experience"]
    lo = df["min_years_experience"]
    hi = df["max_years_experience"]
    df["in_experience_band"] = ((exp >= lo) & (exp <= hi)).astype(int)
    df["overqualified"] = (exp > hi).astype(int)
    df["underqualified"] = (exp < lo).astype(int)
    df["exp_band_distance"] = np.where(exp < lo, lo - exp, np.where(exp > hi, exp - hi, 0))

    cand_sal = df["expected_salary"]
    jmin = df["salary_min_usd"]
    jmax = df["salary_max_usd"]
    df["salary_in_band"] = ((cand_sal >= jmin) & (cand_sal <= jmax)).astype(int)
    df["salary_over_budget"] = (cand_sal > jmax).astype(int)
    df["salary_band_distance"] = np.where(cand_sal < jmin, jmin - cand_sal, np.where(cand_sal > jmax, cand_sal - jmax, 0))

    prev = df["previous_companies"].apply(tokenize_pipe)
    freq = Counter()
    for s in prev:
        freq.update(s)
    df["max_company_freq"] = [max((freq[c] for c in s), default=0) for s in prev]
    df["mean_company_freq"] = [np.mean([freq[c] for c in s]) if s else 0.0 for s in prev]
    df["n_previous_companies"] = prev.apply(len)
    
    df["completeness_x_skill"] = df["profile_completeness"] * df["skill_jaccard"]
    
    # اطمینان از وجود ستون‌ها قبل از محاسبه
    if "in_experience_band" in df.columns:
        df["idf_x_inband"] = df["skill_match_idf"] * df["in_experience_band"]
        
    df["idf_x_salband"] = df["skill_match_idf"] * df["salary_band_distance"]
    df["expgrank_x_salgrank"] = df.groupby('job_id')["experience_gap"].rank(pct=True) * df.groupby('job_id')["salary_gap"].rank(pct=True)
    df["title_x_expgrank"] = df["title_token_jaccard"] * df.groupby('job_id')["experience_gap"].rank(pct=True)
    df["match_per_salovershoot"] = df["skill_match_idf"] / (df["salary_band_distance"] + 1.0)
    df["match_per_expgap"] = df["skill_match_idf"] / (df["exp_band_distance"] + 1.0)

    cols_to_rank = [
        'experience_gap', 'salary_gap', 'skill_match_ratio', 'title_similarity', 
        'days_to_apply', 'english_score', 'edu_score', "idf_x_inband", "idf_x_salband", 
        "expgrank_x_salgrank", "title_x_expgrank", "match_per_salovershoot", "match_per_expgap"
    ]
    g = df.groupby("job_id")
    for c in cols_to_rank:
        if c in df.columns:
            df[f"{c}_grank"] = g[c].rank(pct=True)
            df[f"{c}_gzscore"] = g[c].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))
            
    return df, idf # برگرداندن idf برای استفاده در تست

# نحوه اجرای صحیح:
train_df, train_idf = build_advanced_features(train_df)
test_df, _ = build_advanced_features(test_df, fitted_idf=train_idf)

3

In [17]:
def oof_te_highcard(train, test, cat_col, group_col="job_id", label="relevance_label", n_splits=5, k=20, noise=0.01, min_count=1):
    tr = train.copy()
    te = test.copy()
    
    tr[cat_col] = tr[cat_col].fillna("MISSING").astype(str).str.lower().str.strip()
    te[cat_col] = te[cat_col].fillna("MISSING").astype(str).str.lower().str.strip()

    tr["_nl"] = tr.groupby(group_col)[label].transform(lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
    gmean = tr["_nl"].mean()
    rng = np.random.default_rng(42)
    oof = np.full(len(tr), gmean, dtype=float)
    gkf = GroupKFold(n_splits=n_splits)

    for tr_idx, val_idx in gkf.split(tr, groups=tr[group_col]):
        fold = tr.iloc[tr_idx]
        agg = fold.groupby(cat_col)["_nl"].agg(["mean", "count"])
        agg = agg[agg["count"] >= min_count]
        smooth = (agg["mean"] * agg["count"] + gmean * k) / (agg["count"] + k)
        mapped = tr.iloc[val_idx][cat_col].map(smooth).fillna(gmean).values
        mapped = mapped * (1 + rng.normal(0, noise, size=len(mapped)))
        oof[val_idx] = mapped

    full = tr.groupby(cat_col)["_nl"].agg(["mean", "count"])
    full = full[full["count"] >= min_count]
    full_s = (full["mean"] * full["count"] + gmean * k) / (full["count"] + k)
    test_enc = te[cat_col].map(full_s).fillna(gmean).values
    
    tr_freq = tr[cat_col].map(tr[cat_col].value_counts()).fillna(0)
    te_freq = te[cat_col].map(tr[cat_col].value_counts()).fillna(0)
    
    return oof, test_enc, tr_freq, te_freq

high_card_cols = ['current_title', 'job_location', 'university', 'industry']
for col in high_card_cols:
    tr_enc, te_enc, tr_freq, te_freq = oof_te_highcard(train_df, test_df, col)
    train_df[f"{col}_te"] = tr_enc
    test_df[f"{col}_te"] = te_enc
    train_df[f"{col}_freq"] = tr_freq
    test_df[f"{col}_freq"] = te_freq

3.5

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import paired_cosine_distances

def add_lsa_and_global_features(train_df, test_df):
    text_cols = [('job_title', 'current_title'), ('required_skills', 'skills')]
    
    train_df['global_salary_ratio'] = train_df['expected_salary'] / (train_df['expected_salary'].median() + 1)
    test_df['global_salary_ratio'] = test_df['expected_salary'] / (train_df['expected_salary'].median() + 1)
    
    train_df['profile_strength'] = train_df['english_score'] + train_df['edu_score'] + train_df['n_certifications']
    test_df['profile_strength'] = test_df['english_score'] + test_df['edu_score'] + test_df['n_certifications']

    for col_job, col_cand in text_cols:
        tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000)
        
        corpus_train = train_df[col_job].fillna("") + " " + train_df[col_cand].fillna("")
        tfidf.fit(corpus_train)
        
        job_train_tf = tfidf.transform(train_df[col_job].fillna(""))
        cand_train_tf = tfidf.transform(train_df[col_cand].fillna(""))
        job_test_tf = tfidf.transform(test_df[col_job].fillna(""))
        cand_test_tf = tfidf.transform(test_df[col_cand].fillna(""))
        
        svd = TruncatedSVD(n_components=10, random_state=42)
        job_train_svd = svd.fit_transform(job_train_tf)
        cand_train_svd = svd.transform(cand_train_tf)
        job_test_svd = svd.transform(job_test_tf)
        cand_test_svd = svd.transform(cand_test_tf)
        
        train_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_train_svd, cand_train_svd)
        test_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_test_svd, cand_test_svd)
        
        for i in range(10):
            train_df[f'{col_cand}_svd_{i}'] = cand_train_svd[:, i]
            test_df[f'{col_cand}_svd_{i}'] = cand_test_svd[:, i]
            
    return train_df, test_df

train_df, test_df = add_lsa_and_global_features(train_df, test_df)

/tmp/ipykernel_228815/1098272815.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[f'{col_cand}_svd_{i}'] = cand_train_svd[:, i]
/tmp/ipykernel_228815/1098272815.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[f'{col_cand}_svd_{i}'] = cand_test_svd[:, i]


4


In [19]:
cols_to_drop = [
    'application_id', 'candidate_id', 'job_id', 'application_date', 'job_posted_date',
    'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed', 
    'company_size', 'industry', 'current_title', 'skills', 'education_level', 
    'university', 'previous_companies', 'certifications', 'english_proficiency', 
    'candidate_location', 'willing_to_relocate', 'account_created_date', 'relevance_label'
]
all_features = [c for c in train_df.columns if c not in cols_to_drop and train_df[c].dtype in [np.float64, np.float32, np.int64, np.int32]]

def fast_null_importance_pruner(X, y, groups, features):
    m_actual = lgb.LGBMRanker(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    sort_idx = np.argsort(groups, kind='stable')
    _, counts = np.unique(groups[sort_idx], return_counts=True)
    m_actual.fit(X.iloc[sort_idx], y[sort_idx], group=counts)
    actual_imp = m_actual.booster_.feature_importance(importance_type="gain")
    
    imp_df = pd.DataFrame({'feature': features, 'importance': actual_imp})
    imp_df = imp_df.sort_values('importance', ascending=False)
    keep_features = imp_df.head(50)['feature'].tolist()
    
    return keep_features

final_features = fast_null_importance_pruner(train_df[all_features], train_df['relevance_label'].values, train_df['job_id'].values, all_features)

train_df = train_df.sort_values('job_id').reset_index(drop=True)
test_df = test_df.sort_values('job_id').reset_index(drop=True)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.053530 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16708
[LightGBM] [Info] Number of data points in the train set: 118772, number of used features: 97


5


In [21]:
def ndcg_by_group(y_true, y_pred, groups, k=10):
    df = pd.DataFrame({"t": y_true, "p": y_pred, "g": groups})
    s = [ndcg_score([gr["t"].values], [gr["p"].values], k=k) for _, gr in df.groupby("g") if len(gr) >= 2]
    return float(np.mean(s))

def group_rank(pred, groups):
    return pd.Series(pred).groupby(groups).rank(pct=True).values

X = train_df[final_features]
y = train_df['relevance_label'].values
groups = train_df['job_id'].values

lgb_params = {
    'objective': 'lambdarank', 'metric': 'ndcg', 'n_estimators': 1500,
    'learning_rate': 0.03, 'num_leaves': 31, 'max_depth': 6,
    'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.65,
    'reg_alpha': 0.1, 'reg_lambda': 1.0, 'label_gain': [0, 1, 3, 7, 15],
    'random_state': 42, 'n_jobs': -1
}

job_dates = train_df.groupby('job_id')['application_date'].max().sort_values()
time_sorted_jobs = job_dates.index.to_numpy()
folds = np.array_split(time_sorted_jobs, 5)

cv_splits = []
for i in range(5):
    val_jobs = folds[i]
    tr_jobs = np.concatenate([folds[j] for j in range(5) if j != i])
    
    # تغییر tolist() به to_numpy() برای سازگاری با آرایه‌ها
    tr_idx = train_df.index[train_df['job_id'].isin(tr_jobs)].to_numpy()
    val_idx = train_df.index[train_df['job_id'].isin(val_jobs)].to_numpy()
    
    cv_splits.append((tr_idx, val_idx))

oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

for fold, (tr_idx, val_idx) in enumerate(cv_splits):
    tr_order = np.argsort(groups[tr_idx], kind="stable")
    Xtr, ytr, gtr = X.iloc[tr_idx[tr_order]], y[tr_idx[tr_order]], groups[tr_idx[tr_order]]
    
    va_order = np.argsort(groups[val_idx], kind="stable")
    Xva, yva, gva = X.iloc[val_idx[va_order]], y[val_idx[va_order]], groups[val_idx[va_order]]
    
    _, c_tr = np.unique(gtr, return_counts=True)
    _, c_va = np.unique(gva, return_counts=True)
    
    lm = lgb.LGBMRanker(**lgb_params)
    lm.fit(Xtr, ytr, group=c_tr, eval_set=[(Xva, yva)], eval_group=[c_va], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = lm.predict(X.iloc[val_idx])
    
    cm = CatBoostRanker(loss_function="YetiRank", eval_metric="NDCG:top=10", iterations=1000, learning_rate=0.03, depth=6, l2_leaf_reg=3, random_seed=42, verbose=0, early_stopping_rounds=50)
    cm.fit(Pool(Xtr, ytr, group_id=gtr), eval_set=Pool(Xva, yva, group_id=gva))
    oof_cat[val_idx] = cm.predict(X.iloc[val_idx])

r_lgb = group_rank(oof_lgb, groups)
r_cat = group_rank(oof_cat, groups)

best_score, best_w = -1, 0.5
for w in np.linspace(0, 1, 21):
    sc = ndcg_by_group(y, w * r_lgb + (1 - w) * r_cat, groups)
    if sc > best_score:
        best_score, best_w = sc, w

_, c_full = np.unique(groups, return_counts=True)

final_lgb = lgb.LGBMRanker(**lgb_params)
final_lgb.fit(X, y, group=c_full)

final_cat = CatBoostRanker(loss_function="YetiRank", iterations=1000, learning_rate=0.03, depth=6, l2_leaf_reg=3, random_seed=42, verbose=0)
final_cat.fit(Pool(X, y, group_id=groups))

X_test = test_df[final_features]
test_groups = test_df['job_id'].values

final_blend = best_w * group_rank(final_lgb.predict(X_test), test_groups) + (1 - best_w) * group_rank(final_cat.predict(X_test), test_groups)

sub = pd.DataFrame({'application_id': test_df['application_id'], 'score': final_blend})
sub = sub.sort_values('application_id').reset_index(drop=True)
sub.to_csv('submission_ensemble_final.csv', index=False)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.066045 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9091
[LightGBM] [Info] Number of data points in the train set: 101108, number of used features: 50
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018399 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9082
[LightGBM] [Info] Number of data points in the train set: 95455, number of used features: 50
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.115902 seconds.
You can set `force_col_wise=true` to re

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import lightgbm as lgb
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import ndcg_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import paired_cosine_distances
import difflib

train_app = pd.read_csv('applications_train.csv')
test_app = pd.read_csv('applications_test.csv')
jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')

train_df = train_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
test_df = test_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')

eng_map = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
edu_map = {'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
exchange_rates = {'USD': 1.0, 'EUR': 1.08, 'IRR': 0.00002}

for df in [train_df, test_df]:
    df['english_score'] = df['english_proficiency'].map(eng_map).fillna(0)
    df['edu_score'] = df['education_level'].map(edu_map).fillna(0)
    
    df['application_date'] = pd.to_datetime(df['application_date'], dayfirst=True, format='mixed')
    df['job_posted_date'] = pd.to_datetime(df['job_posted_date'], dayfirst=True, format='mixed')
    df['days_to_apply'] = (df['application_date'] - df['job_posted_date']).dt.days.fillna(0)
    
    rates = df['salary_currency'].map(exchange_rates).fillna(1.0)
    df['salary_min_usd'] = df['salary_min'] * rates
    df['salary_max_usd'] = df['salary_max'] * rates
    
    df['expected_salary'] = df['expected_salary'].fillna(df['expected_salary'].median())
    df['salary_min_usd'] = df['salary_min_usd'].fillna(df['salary_min_usd'].median())
    df['salary_max_usd'] = df['salary_max_usd'].fillna(df['salary_max_usd'].median())
    
    df['salary_gap'] = df['expected_salary'] - ((df['salary_min_usd'] + df['salary_max_usd']) / 2)
    df['experience_gap'] = df['years_experience'] - df['min_years_experience']
    
    df['skill_match_ratio'] = df.apply(
        lambda row: len(set(str(row['skills']).split('|')) & set(str(row['required_skills']).split('|'))) / 
                    max(1, len(set(str(row['required_skills']).split('|')))), axis=1
    )
    df['title_similarity'] = df.apply(
        lambda row: difflib.SequenceMatcher(None, str(row['job_title']).lower(), str(row['current_title']).lower()).ratio(), axis=1
    )

def tokenize_pipe(s):
    if pd.isna(s):
        return set()
    return {t.strip().lower() for t in str(s).split("|") if t.strip()}

def tokenize_title(s):
    if pd.isna(s):
        return set()
    return {w for w in str(s).lower().replace(".", " ").split() if len(w) > 1}

def build_advanced_features(df, fitted_idf=None):
    cand_sk = df["skills"].apply(tokenize_pipe)
    req_sk = df["required_skills"].apply(tokenize_pipe)
    certs = df["certifications"].apply(tokenize_pipe)
    job_tok = df["job_title"].apply(tokenize_title)
    cur_tok = df["current_title"].apply(tokenize_title)

    df["skill_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(cand_sk, req_sk)]
    df["missing_required_skills"] = [len(b - a) for a, b in zip(cand_sk, req_sk)]
    df["extra_skills"] = [len(a - b) for a, b in zip(cand_sk, req_sk)]
    df["cert_skill_overlap"] = [len(c & b) for c, b in zip(certs, req_sk)]
    df["cert_title_overlap"] = [len(c & j) for c, j in zip(certs, job_tok)]
    df["n_certifications"] = certs.apply(len)
    df["title_token_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(job_tok, cur_tok)]

    if fitted_idf is None:
        dfreq = Counter()
        for s in req_sk:
            dfreq.update(s)
        n_docs = len(req_sk)
        idf = {k: np.log((1 + n_docs) / (1 + v)) + 1 for k, v in dfreq.items()}
    else:
        idf = fitted_idf

    vals = []
    for a, b in zip(cand_sk, req_sk):
        if not b:
            vals.append(0.0)
            continue
        num = sum(idf.get(t, 0.0) for t in (a & b))
        den = sum(idf.get(t, 0.0) for t in b)
        vals.append(num / den if den else 0.0)
    df["skill_match_idf"] = vals

    exp = df["years_experience"]
    lo = df["min_years_experience"]
    hi = df["max_years_experience"]
    df["in_experience_band"] = ((exp >= lo) & (exp <= hi)).astype(int)
    df["overqualified"] = (exp > hi).astype(int)
    df["underqualified"] = (exp < lo).astype(int)
    df["exp_band_distance"] = np.where(exp < lo, lo - exp, np.where(exp > hi, exp - hi, 0))

    cand_sal = df["expected_salary"]
    jmin = df["salary_min_usd"]
    jmax = df["salary_max_usd"]
    df["salary_in_band"] = ((cand_sal >= jmin) & (cand_sal <= jmax)).astype(int)
    df["salary_over_budget"] = (cand_sal > jmax).astype(int)
    df["salary_band_distance"] = np.where(cand_sal < jmin, jmin - cand_sal, np.where(cand_sal > jmax, cand_sal - jmax, 0))

    prev = df["previous_companies"].apply(tokenize_pipe)
    freq = Counter()
    for s in prev:
        freq.update(s)
    df["max_company_freq"] = [max((freq[c] for c in s), default=0) for s in prev]
    df["mean_company_freq"] = [np.mean([freq[c] for c in s]) if s else 0.0 for s in prev]
    df["n_previous_companies"] = prev.apply(len)
    
    df["completeness_x_skill"] = df["profile_completeness"] * df["skill_jaccard"]
    
    if "in_experience_band" in df.columns:
        df["idf_x_inband"] = df["skill_match_idf"] * df["in_experience_band"]
        
    df["idf_x_salband"] = df["skill_match_idf"] * df["salary_band_distance"]
    df["expgrank_x_salgrank"] = df.groupby('job_id')["experience_gap"].rank(pct=True) * df.groupby('job_id')["salary_gap"].rank(pct=True)
    df["title_x_expgrank"] = df["title_token_jaccard"] * df.groupby('job_id')["experience_gap"].rank(pct=True)
    df["match_per_salovershoot"] = df["skill_match_idf"] / (df["salary_band_distance"] + 1.0)
    df["match_per_expgap"] = df["skill_match_idf"] / (df["exp_band_distance"] + 1.0)

    cols_to_rank = [
        'experience_gap', 'salary_gap', 'skill_match_ratio', 'title_similarity', 
        'days_to_apply', 'english_score', 'edu_score', "idf_x_inband", "idf_x_salband", 
        "expgrank_x_salgrank", "title_x_expgrank", "match_per_salovershoot", "match_per_expgap"
    ]
    g = df.groupby("job_id")
    for c in cols_to_rank:
        if c in df.columns:
            df[f"{c}_grank"] = g[c].rank(pct=True)
            df[f"{c}_gzscore"] = g[c].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))
            
    return df, idf

def add_ltr_magic_features(df):
    g = df.groupby('job_id')
    for col in ['years_experience', 'expected_salary', 'skill_match_ratio', 'profile_completeness']:
        df[f'{col}_query_zscore'] = g[col].transform(lambda x: (x - x.mean()) / (x.std() + 1e-5))
        df[f'{col}_query_max_ratio'] = df[col] / (g[col].transform('max') + 1e-5)

    df['competitor_count'] = g['candidate_id'].transform('count')
    df['skill_len'] = df['skills'].astype(str).apply(lambda x: len(x.split('|')))
    df['avg_skills_in_query'] = g['skill_len'].transform('mean')
    df['skill_hoarding_ratio'] = df['skill_len'] / (df['avg_skills_in_query'] + 1e-5)
    df['value_for_money_idx'] = df['years_experience_query_zscore'] - df['expected_salary_query_zscore']
    return df

def add_lsa_and_global_features(train_df, test_df):
    text_cols = [('job_title', 'current_title'), ('required_skills', 'skills')]
    
    train_df['global_salary_ratio'] = train_df['expected_salary'] / (train_df['expected_salary'].median() + 1)
    test_df['global_salary_ratio'] = test_df['expected_salary'] / (train_df['expected_salary'].median() + 1)
    
    train_df['profile_strength'] = train_df['english_score'] + train_df['edu_score'] + train_df['n_certifications']
    test_df['profile_strength'] = test_df['english_score'] + test_df['edu_score'] + test_df['n_certifications']

    for col_job, col_cand in text_cols:
        tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000)
        corpus_train = train_df[col_job].fillna("") + " " + train_df[col_cand].fillna("")
        tfidf.fit(corpus_train)
        
        job_train_tf = tfidf.transform(train_df[col_job].fillna(""))
        cand_train_tf = tfidf.transform(train_df[col_cand].fillna(""))
        job_test_tf = tfidf.transform(test_df[col_job].fillna(""))
        cand_test_tf = tfidf.transform(test_df[col_cand].fillna(""))
        
        svd = TruncatedSVD(n_components=10, random_state=42)
        job_train_svd = svd.fit_transform(job_train_tf)
        cand_train_svd = svd.transform(cand_train_tf)
        job_test_svd = svd.transform(job_test_tf)
        cand_test_svd = svd.transform(cand_test_tf)
        
        train_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_train_svd, cand_train_svd)
        test_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_test_svd, cand_test_svd)
        
        for i in range(10):
            train_df[f'{col_cand}_svd_{i}'] = cand_train_svd[:, i]
            test_df[f'{col_cand}_svd_{i}'] = cand_test_svd[:, i]
            
    return train_df, test_df

train_df, train_idf = build_advanced_features(train_df)
test_df, _ = build_advanced_features(test_df, fitted_idf=train_idf)
train_df = add_ltr_magic_features(train_df)
test_df = add_ltr_magic_features(test_df)
train_df, test_df = add_lsa_and_global_features(train_df, test_df)

train_df = train_df.sort_values('job_id').reset_index(drop=True)
test_df = test_df.sort_values('job_id').reset_index(drop=True)

job_dates = train_df.groupby('job_id')['application_date'].max().sort_values()
time_sorted_jobs = job_dates.index.to_numpy()
folds = np.array_split(time_sorted_jobs, 5)

cv_splits = []
for i in range(4): 
    tr_jobs = np.concatenate(folds[:i+1])
    val_jobs = folds[i+1]
    tr_idx = train_df.index[train_df['job_id'].isin(tr_jobs)].to_numpy()
    val_idx = train_df.index[train_df['job_id'].isin(val_jobs)].to_numpy()
    cv_splits.append((tr_idx, val_idx))

def temporal_oof_te(train, test, cat_col, splits, label="relevance_label", k=20):
    tr = train.copy()
    te = test.copy()
    gmean = tr[label].mean()
    oof = np.full(len(tr), gmean, dtype=float)
    
    for tr_idx, val_idx in splits:
        tr_fold = tr.iloc[tr_idx]
        val_fold = tr.iloc[val_idx]
        agg = tr_fold.groupby(cat_col)[label].agg(["mean", "count"])
        smooth = (agg["mean"] * agg["count"] + gmean * k) / (agg["count"] + k)
        oof[val_idx] = val_fold[cat_col].map(smooth).fillna(gmean).values
        
    full_agg = tr.groupby(cat_col)[label].agg(["mean", "count"])
    full_smooth = (full_agg["mean"] * full_agg["count"] + gmean * k) / (full_agg["count"] + k)
    test_enc = te[cat_col].map(full_smooth).fillna(gmean).values
    return oof, test_enc

high_card_cols = ['current_title', 'job_location', 'university', 'industry']
for col in high_card_cols:
    train_df[col] = train_df[col].fillna("MISSING").astype(str).str.lower().str.strip()
    test_df[col] = test_df[col].fillna("MISSING").astype(str).str.lower().str.strip()
    tr_enc, te_enc = temporal_oof_te(train_df, test_df, col, cv_splits)
    train_df[f"{col}_te"] = tr_enc
    test_df[f"{col}_te"] = te_enc
    tr_freq = train_df[col].map(train_df[col].value_counts()).fillna(0)
    te_freq = test_df[col].map(train_df[col].value_counts()).fillna(0)
    train_df[f"{col}_freq"] = tr_freq
    test_df[f"{col}_freq"] = te_freq

cols_to_drop = [
    'application_id', 'candidate_id', 'job_id', 'application_date', 'job_posted_date',
    'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed', 
    'company_size', 'industry', 'current_title', 'skills', 'education_level', 
    'university', 'previous_companies', 'certifications', 'english_proficiency', 
    'candidate_location', 'willing_to_relocate', 'account_created_date', 'relevance_label'
]
all_features = [c for c in train_df.columns if c not in cols_to_drop and train_df[c].dtype in [np.float64, np.float32, np.int64, np.int32]]

def fast_null_importance_pruner(X, y, groups, features):
    m_actual = lgb.LGBMRanker(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    sort_idx = np.argsort(groups, kind='stable')
    _, counts = np.unique(groups[sort_idx], return_counts=True)
    m_actual.fit(X.iloc[sort_idx], y[sort_idx], group=counts)
    imp_df = pd.DataFrame({'feature': features, 'importance': m_actual.booster_.feature_importance(importance_type="gain")})
    return imp_df.sort_values('importance', ascending=False).head(50)['feature'].tolist()

final_features = fast_null_importance_pruner(train_df[all_features], train_df['relevance_label'].values, train_df['job_id'].values, all_features)

X = train_df[final_features]
y = train_df['relevance_label'].values
groups = train_df['job_id'].values

lgb_params = {
    'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [5, 10], 'lambdarank_truncation_level': 15,
    'n_estimators': 2000, 'learning_rate': 0.015, 'num_leaves': 63, 'max_depth': -1,
    'min_child_samples': 3, 'subsample': 0.8, 'colsample_bytree': 0.85,
    'reg_alpha': 0.5, 'reg_lambda': 2.0, 'label_gain': [0, 1, 3, 7, 15],
    'random_state': 42, 'n_jobs': -1
}

oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

def ndcg_by_group(y_true, y_pred, groups, k=10):
    df = pd.DataFrame({"t": y_true, "p": y_pred, "g": groups})
    s = [ndcg_score([gr["t"].values], [gr["p"].values], k=k) for _, gr in df.groupby("g") if len(gr) >= 2]
    return float(np.mean(s))

def group_rank(pred, groups):
    return pd.Series(pred).groupby(groups).rank(pct=True).values

def reciprocal_rank_fusion(r1, r2, k=60, w=0.5):
    pos_r1 = len(r1) - (r1 * len(r1))
    pos_r2 = len(r2) - (r2 * len(r2))
    return w * (1.0 / (pos_r1 + k)) + (1.0 - w) * (1.0 / (pos_r2 + k))

for fold, (tr_idx, val_idx) in enumerate(cv_splits):
    tr_order = np.argsort(groups[tr_idx], kind="stable")
    Xtr, ytr, gtr = X.iloc[tr_idx[tr_order]], y[tr_idx[tr_order]], groups[tr_idx[tr_order]]
    va_order = np.argsort(groups[val_idx], kind="stable")
    Xva, yva, gva = X.iloc[val_idx[va_order]], y[val_idx[va_order]], groups[val_idx[va_order]]
    _, c_tr = np.unique(gtr, return_counts=True)
    _, c_va = np.unique(gva, return_counts=True)
    
    lm = lgb.LGBMRanker(**lgb_params)
    lm.fit(Xtr, ytr, group=c_tr, eval_set=[(Xva, yva)], eval_group=[c_va], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = lm.predict(X.iloc[val_idx])
    
    cm = CatBoostRanker(loss_function="YetiRank", eval_metric="NDCG:top=10", iterations=1000, learning_rate=0.03, depth=6, l2_leaf_reg=3, random_seed=42, verbose=0, early_stopping_rounds=50)
    cm.fit(Pool(Xtr, ytr, group_id=gtr), eval_set=Pool(Xva, yva, group_id=gva))
    oof_cat[val_idx] = cm.predict(X.iloc[val_idx])

val_mask = np.concatenate([v for _, v in cv_splits])
r_lgb_val = group_rank(oof_lgb[val_mask], groups[val_mask])
r_cat_val = group_rank(oof_cat[val_mask], groups[val_mask])
y_val = y[val_mask]
groups_val = groups[val_mask]

best_score, best_w = -1, 0.5
for w in np.linspace(0, 1, 21):
    fused = reciprocal_rank_fusion(r_lgb_val, r_cat_val, k=60, w=w)
    sc = ndcg_by_group(y_val, fused, groups_val)
    if sc > best_score:
        best_score, best_w = sc, w

_, c_full = np.unique(groups, return_counts=True)

final_lgb = lgb.LGBMRanker(**lgb_params)
final_lgb.fit(X, y, group=c_full)

final_cat = CatBoostRanker(loss_function="YetiRank", iterations=1000, learning_rate=0.03, depth=6, l2_leaf_reg=3, random_seed=42, verbose=0)
final_cat.fit(Pool(X, y, group_id=groups))

X_test = test_df[final_features]
test_groups = test_df['job_id'].values

final_r_lgb = group_rank(final_lgb.predict(X_test), test_groups)
final_r_cat = group_rank(final_cat.predict(X_test), test_groups)
sub_scores = reciprocal_rank_fusion(final_r_lgb, final_r_cat, k=60, w=best_w)

sub = pd.DataFrame({'application_id': test_df['application_id'], 'score': sub_scores})
sub = sub.sort_values('application_id').reset_index(drop=True)
sub.to_csv('submission_ensemble_final3.csv', index=False)

/tmp/ipykernel_286949/3233640512.py:185: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[f'{col_cand}_svd_{i}'] = cand_train_svd[:, i]
/tmp/ipykernel_286949/3233640512.py:186: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[f'{col_cand}_svd_{i}'] = cand_test_svd[:, i]
/tmp/ipykernel_286949/3233640512.py:234: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) ins

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.053276 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 19026
[LightGBM] [Info] Number of data points in the train set: 118772, number of used features: 110


/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004959 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8511
[LightGBM] [Info] Number of data points in the train set: 17664, number of used features: 49


/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


...

/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034762 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8794
[LightGBM] [Info] Number of data points in the train set: 88778, number of used features: 50


/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019269 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8848
[LightGBM] [Info] Number of data points in the train set: 118772, number of used features: 50


/media/E/term6/DS/Datasets/task 3 - datasets/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


In [ ]:
"""
Learning-to-Rank pipeline — corrected build.

Run order is top to bottom. Key changes vs the previous version:
  1. Raw SVD components dropped (basis invalid on cold-start jobs); only
     sign-invariant similarity / dot-product summaries kept.
  2. Target Encoding moved INSIDE the model CV loop so it never leaks across
     the same split the model is scored on. Global TE removed.
  3. Real null-importance pruner (labels shuffled within groups) replaces the
     actual-importance-only "pruner".
  4. Three magic feature families added; four previously-discarded location
     columns now used.
  5. lambdarank_truncation_level + relaxed colsample/min_split_gain to fix the
     "No further splits" warnings and concentrate gradient on the top.
  6. Final models reuse the median early-stopped iteration count; blend weight
     fit per segment (seen vs cold-start) with a power transform.

CV CHOICE: symmetric job-fold split (not expanding-window). This is the safer
default when unseen test jobs may be temporally interleaved with train. If you
confirm unseen jobs are strictly LATER, swap in the expanding-window block
marked below.
"""

import numpy as np
import pandas as pd
from collections import Counter
import difflib

import lightgbm as lgb
from catboost import CatBoostRanker, Pool
from sklearn.metrics import ndcg_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import paired_cosine_distances

RNG_SEED = 42

train_app = pd.read_csv('applications_train.csv')
test_app = pd.read_csv('applications_test.csv')
jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')

train_df = train_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
test_df = test_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')

train_job_ids = set(train_df['job_id'].unique())

eng_map = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
edu_map = {'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
exchange_rates = {'USD': 1.0, 'EUR': 1.08, 'IRR': 0.00002}

def _truthy(s):
    return s.fillna(0).astype(str).str.lower().isin(['1', 'true', 'yes', 'y', 't'])

_exp_sal_med = None
_sal_min_med = None
_sal_max_med = None

for i, df in enumerate([train_df, test_df]):
    df['english_score'] = df['english_proficiency'].map(eng_map).fillna(0)
    df['edu_score'] = df['education_level'].map(edu_map).fillna(0)

    df['application_date'] = pd.to_datetime(df['application_date'], dayfirst=True, format='mixed')
    df['job_posted_date'] = pd.to_datetime(df['job_posted_date'], dayfirst=True, format='mixed')
    df['days_to_apply'] = (df['application_date'] - df['job_posted_date']).dt.days.fillna(0)

    rates = df['salary_currency'].map(exchange_rates).fillna(1.0)
    df['salary_min_usd'] = df['salary_min'] * rates
    df['salary_max_usd'] = df['salary_max'] * rates

    if i == 0: 
        _exp_sal_med = df['expected_salary'].median()
        _sal_min_med = df['salary_min_usd'].median()
        _sal_max_med = df['salary_max_usd'].median()

    df['expected_salary'] = df['expected_salary'].fillna(_exp_sal_med)
    df['salary_min_usd'] = df['salary_min_usd'].fillna(_sal_min_med)
    df['salary_max_usd'] = df['salary_max_usd'].fillna(_sal_max_med)

    df['salary_gap'] = df['expected_salary'] - ((df['salary_min_usd'] + df['salary_max_usd']) / 2)
    df['experience_gap'] = df['years_experience'] - df['min_years_experience']

    df['skill_match_ratio'] = df.apply(
        lambda row: len(set(str(row['skills']).split('|')) & set(str(row['required_skills']).split('|'))) /
                    max(1, len(set(str(row['required_skills']).split('|')))), axis=1
    )
    df['title_similarity'] = df.apply(
        lambda row: difflib.SequenceMatcher(
            None, str(row['job_title']).lower(), str(row['current_title']).lower()).ratio(), axis=1
    )



def tokenize_pipe(s):
    if pd.isna(s):
        return set()
    return {t.strip().lower() for t in str(s).split("|") if t.strip()}


def tokenize_title(s):
    if pd.isna(s):
        return set()
    return {w for w in str(s).lower().replace(".", " ").split() if len(w) > 1}



def build_advanced_features(df, fitted_idf=None, fitted_company_freq=None):
    cand_sk = df["skills"].apply(tokenize_pipe)
    req_sk = df["required_skills"].apply(tokenize_pipe)
    certs = df["certifications"].apply(tokenize_pipe)
    job_tok = df["job_title"].apply(tokenize_title)
    cur_tok = df["current_title"].apply(tokenize_title)

    df["skill_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(cand_sk, req_sk)]
    df["missing_required_skills"] = [len(b - a) for a, b in zip(cand_sk, req_sk)]
    df["extra_skills"] = [len(a - b) for a, b in zip(cand_sk, req_sk)]
    df["cert_skill_overlap"] = [len(c & b) for c, b in zip(certs, req_sk)]
    df["cert_title_overlap"] = [len(c & j) for c, j in zip(certs, job_tok)]
    df["n_certifications"] = certs.apply(len)
    df["title_token_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(job_tok, cur_tok)]

    if fitted_idf is None:
        dfreq = Counter()
        for s in req_sk:
            dfreq.update(s)
        n_docs = len(req_sk)
        idf = {k: np.log((1 + n_docs) / (1 + v)) + 1 for k, v in dfreq.items()}
    else:
        idf = fitted_idf

    vals = []
    for a, b in zip(cand_sk, req_sk):
        if not b:
            vals.append(0.0)
            continue
        num = sum(idf.get(t, 0.0) for t in (a & b))
        den = sum(idf.get(t, 0.0) for t in b)
        vals.append(num / den if den else 0.0)
    df["skill_match_idf"] = vals

    exp = df["years_experience"]
    lo = df["min_years_experience"]
    hi = df["max_years_experience"]
    df["in_experience_band"] = ((exp >= lo) & (exp <= hi)).astype(int)
    df["overqualified"] = (exp > hi).astype(int)
    df["underqualified"] = (exp < lo).astype(int)
    df["exp_band_distance"] = np.where(exp < lo, lo - exp, np.where(exp > hi, exp - hi, 0))

    cand_sal = df["expected_salary"]
    jmin = df["salary_min_usd"]
    jmax = df["salary_max_usd"]
    df["salary_in_band"] = ((cand_sal >= jmin) & (cand_sal <= jmax)).astype(int)
    df["salary_over_budget"] = (cand_sal > jmax).astype(int)
    df["salary_band_distance"] = np.where(cand_sal < jmin, jmin - cand_sal,
                                          np.where(cand_sal > jmax, cand_sal - jmax, 0))

    prev = df["previous_companies"].apply(tokenize_pipe)
    if fitted_company_freq is None:
        freq = Counter()
        for s in prev:
            freq.update(s)
    else:
        freq = fitted_company_freq
    df["max_company_freq"] = [max((freq[c] for c in s), default=0) for s in prev]
    df["mean_company_freq"] = [np.mean([freq[c] for c in s]) if s else 0.0 for s in prev]
    df["n_previous_companies"] = prev.apply(len)

    df["completeness_x_skill"] = df["profile_completeness"] * df["skill_jaccard"]
    df["idf_x_inband"] = df["skill_match_idf"] * df["in_experience_band"]
    df["idf_x_salband"] = df["skill_match_idf"] * df["salary_band_distance"]

    g = df.groupby('job_id')
    df["expgrank_x_salgrank"] = g["experience_gap"].rank(pct=True) * g["salary_gap"].rank(pct=True)
    df["title_x_expgrank"] = df["title_token_jaccard"] * g["experience_gap"].rank(pct=True)
    df["match_per_salovershoot"] = df["skill_match_idf"] / (df["salary_band_distance"] + 1.0)
    df["match_per_expgap"] = df["skill_match_idf"] / (df["exp_band_distance"] + 1.0)

    rs = g["skill_match_idf"].rank(ascending=False, method="min")
    re = g["exp_band_distance"].rank(ascending=True, method="min")   # lower distance = better
    df["consensus_rr"] = 1.0 / (rs + 1.0) + 1.0 / (re + 1.0)
    df["rank_spread"] = (rs - re).abs()

    df["skill_z"] = g["skill_match_idf"].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))
    df["skill_gap_to_best"] = g["skill_match_idf"].transform("max") - df["skill_match_idf"]
    df["idf_gap_to_best"] = g["skill_match_idf"].transform("max") - df["skill_match_idf"]

    same_loc = (df["candidate_location"].fillna("").astype(str).str.lower() ==
                df["job_location"].fillna("").astype(str).str.lower()).astype(int)
    remote_ok = _truthy(df["remote_allowed"]) if "remote_allowed" in df.columns else pd.Series(False, index=df.index)
    relocate_ok = _truthy(df["willing_to_relocate"]) if "willing_to_relocate" in df.columns else pd.Series(False, index=df.index)
    df["loc_same"] = same_loc
    df["loc_compatible"] = ((same_loc == 1) | remote_ok | relocate_ok).astype(int)
    df["loc_friction"] = ((1 - same_loc) * (~remote_ok).astype(int)).astype(int)

    cols_to_rank = [
        'experience_gap', 'salary_gap', 'skill_match_ratio', 'title_similarity',
        'days_to_apply', 'english_score', 'edu_score', "idf_x_inband", "idf_x_salband",
        "expgrank_x_salgrank", "title_x_expgrank", "match_per_salovershoot", "match_per_expgap",
        "consensus_rr", "skill_gap_to_best", "loc_compatible",
    ]
    for c in cols_to_rank:
        if c in df.columns:
            df[f"{c}_grank"] = g[c].rank(pct=True)
            df[f"{c}_gzscore"] = g[c].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))

    return df, idf, freq


train_df, train_idf, train_company_freq = build_advanced_features(train_df)
test_df, _, _ = build_advanced_features(test_df, fitted_idf=train_idf, fitted_company_freq=train_company_freq)



def add_lsa_and_global_features(train_df, test_df):
    text_cols = [('job_title', 'current_title'), ('required_skills', 'skills')]

    med = train_df['expected_salary'].median()
    train_df['global_salary_ratio'] = train_df['expected_salary'] / (med + 1)
    test_df['global_salary_ratio'] = test_df['expected_salary'] / (med + 1)

    train_df['profile_strength'] = train_df['english_score'] + train_df['edu_score'] + train_df['n_certifications']
    test_df['profile_strength'] = test_df['english_score'] + test_df['edu_score'] + test_df['n_certifications']

    for col_job, col_cand in text_cols:
        tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000)
        corpus_train = train_df[col_job].fillna("") + " " + train_df[col_cand].fillna("")
        tfidf.fit(corpus_train)

        job_tr = tfidf.transform(train_df[col_job].fillna(""))
        cand_tr = tfidf.transform(train_df[col_cand].fillna(""))
        job_te = tfidf.transform(test_df[col_job].fillna(""))
        cand_te = tfidf.transform(test_df[col_cand].fillna(""))

        svd = TruncatedSVD(n_components=10, random_state=RNG_SEED)
        job_tr_s = svd.fit_transform(job_tr)
        cand_tr_s = svd.transform(cand_tr)
        job_te_s = svd.transform(job_te)
        cand_te_s = svd.transform(cand_te)

        train_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_tr_s, cand_tr_s)
        test_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_te_s, cand_te_s)

        prod_tr = job_tr_s * cand_tr_s
        prod_te = job_te_s * cand_te_s
        train_df[f'{col_job}_svd_dotmax'] = prod_tr.max(axis=1)
        train_df[f'{col_job}_svd_dotsum'] = prod_tr.sum(axis=1)
        test_df[f'{col_job}_svd_dotmax'] = prod_te.max(axis=1)
        test_df[f'{col_job}_svd_dotsum'] = prod_te.sum(axis=1)

    return train_df, test_df


train_df, test_df = add_lsa_and_global_features(train_df, test_df)



HIGH_CARD_COLS = ['current_title', 'job_location', 'university', 'industry']
TE_K = 20
TE_NOISE = 0.01
TE_MIN_COUNT = 1


def _norm_cat(series):
    return series.fillna("MISSING").astype(str).str.lower().str.strip()


for c in HIGH_CARD_COLS:
    train_df[c + "_normcat"] = _norm_cat(train_df[c])
    test_df[c + "_normcat"] = _norm_cat(test_df[c])

train_df["_nl_full"] = train_df.groupby("job_id")["relevance_label"].transform(
    lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
_global_gmean = train_df["_nl_full"].mean()

test_te_maps = {}
test_freq_maps = {}
for c in HIGH_CARD_COLS:
    agg = train_df.groupby(c + "_normcat")["_nl_full"].agg(["mean", "count"])
    agg = agg[agg["count"] >= TE_MIN_COUNT]
    smooth = (agg["mean"] * agg["count"] + _global_gmean * TE_K) / (agg["count"] + TE_K)
    test_te_maps[c] = smooth
    test_freq_maps[c] = train_df[c + "_normcat"].value_counts()


def fit_te_on_fold(fold_train_df, cat_col, gmean):
    """Smoothed target-encoding map from a fold's TRAIN rows only."""
    agg = fold_train_df.groupby(cat_col + "_normcat")["_nl_fold"].agg(["mean", "count"])
    agg = agg[agg["count"] >= TE_MIN_COUNT]
    return (agg["mean"] * agg["count"] + gmean * TE_K) / (agg["count"] + TE_K)



cols_to_drop = [
    'application_id', 'candidate_id', 'job_id', 'application_date', 'job_posted_date',
    'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed',
    'company_size', 'industry', 'current_title', 'skills', 'education_level',
    'university', 'previous_companies', 'certifications', 'english_proficiency',
    'candidate_location', 'willing_to_relocate', 'account_created_date', 'relevance_label',
    '_nl_full', '_nl_fold',
] + [c + "_normcat" for c in HIGH_CARD_COLS]

for c in HIGH_CARD_COLS:
    train_df[f"{c}_te"] = np.nan
    train_df[f"{c}_freq"] = 0.0
    test_df[f"{c}_te"] = test_df[c + "_normcat"].map(test_te_maps[c]).fillna(_global_gmean).values
    test_df[f"{c}_freq"] = test_df[c + "_normcat"].map(test_freq_maps[c]).fillna(0).values

candidate_features = [
    c for c in train_df.columns
    if c not in cols_to_drop and train_df[c].dtype in [np.float64, np.float32, np.int64, np.int32]
]


def null_importance_pruner(X, y, groups, features, n_runs=15, pct=75, seed=0):
    sort_idx = np.argsort(groups, kind='stable')
    Xs = X.iloc[sort_idx].reset_index(drop=True)
    ys = y[sort_idx].copy()
    _, counts = np.unique(groups[sort_idx], return_counts=True)

    base = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, num_leaves=63,
                          random_state=seed, n_jobs=-1)
    base.fit(Xs, ys, group=counts)
    actual = base.booster_.feature_importance("gain")

    null = np.zeros((n_runs, len(features)))
    for r in range(n_runs):
        yp = ys.copy()
        start = 0
        rs = np.random.RandomState(1000 + r)
        for cnt in counts:
            block = yp[start:start + cnt].copy()
            rs.shuffle(block)
            yp[start:start + cnt] = block
            start += cnt
        m = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, num_leaves=63,
                           random_state=r, n_jobs=-1)
        m.fit(Xs, yp, group=counts)
        null[r] = m.booster_.feature_importance("gain")

    thresh = np.percentile(null, pct, axis=0)
    score = np.log1p(actual) - np.log1p(thresh)
    keep = [f for f, s in zip(features, score) if s > 0]
    if len(keep) < 15:
        order = np.argsort(actual)[::-1]
        keep = [features[i] for i in order[:50]]
    return keep



te_cols = [f"{c}_te" for c in HIGH_CARD_COLS] + [f"{c}_freq" for c in HIGH_CARD_COLS]
static_features = [c for c in candidate_features if c not in te_cols]

kept_static = null_importance_pruner(
    train_df[static_features],
    train_df['relevance_label'].values,
    train_df['job_id'].values,
    static_features,
)
final_features = kept_static + te_cols
print(f"[pruner] kept {len(kept_static)} static + {len(te_cols)} TE = {len(final_features)} features")

train_df = train_df.sort_values('job_id').reset_index(drop=True)
test_df = test_df.sort_values('job_id').reset_index(drop=True)


def ndcg_by_group(y_true, y_pred, groups, k=10):
    d = pd.DataFrame({"t": y_true, "p": y_pred, "g": groups})
    s = [ndcg_score([gr["t"].values], [gr["p"].values], k=k)
         for _, gr in d.groupby("g") if len(gr) >= 2]
    return float(np.mean(s))


def group_rank(pred, groups):
    return pd.Series(pred).groupby(groups).rank(pct=True).values


y = train_df['relevance_label'].values
groups = train_df['job_id'].values

lgb_params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'eval_at': [10],
    'lambdarank_truncation_level': 12,   
    'n_estimators': 3000,
    'learning_rate': 0.02,
    'num_leaves': 63,
    'max_depth': 7,
    'min_child_samples': 30,
    'min_split_gain': 0.0,               
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,           
    'reg_alpha': 0.5,
    'reg_lambda': 5.0,
    'label_gain': [0, 1, 3, 7, 15],
    'random_state': RNG_SEED,
    'n_jobs': -1,
    'verbosity': -1,
}

cat_params = dict(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_seed=RNG_SEED,
    verbose=0,
)

job_dates = train_df.groupby('job_id')['application_date'].max().sort_values()
time_sorted_jobs = job_dates.index.to_numpy()
folds = np.array_split(time_sorted_jobs, 5)

cv_splits = []
for i in range(5):
    val_jobs = folds[i]
    tr_jobs = np.concatenate([folds[j] for j in range(5) if j != i])
    tr_idx = train_df.index[train_df['job_id'].isin(tr_jobs)].to_numpy()
    val_idx = train_df.index[train_df['job_id'].isin(val_jobs)].to_numpy()
    cv_splits.append((tr_idx, val_idx))

# cv_splits = []
# for i in range(1, 5):
#     tr_jobs = np.concatenate(folds[:i])      # past only
#     val_jobs = folds[i]                      # future
#     tr_idx = train_df.index[train_df['job_id'].isin(tr_jobs)].to_numpy()
#     val_idx = train_df.index[train_df['job_id'].isin(val_jobs)].to_numpy()
#     cv_splits.append((tr_idx, val_idx))

oof_lgb = np.full(len(train_df), np.nan)
oof_cat = np.full(len(train_df), np.nan)
lgb_best_iters, cat_best_iters = [], []
rng = np.random.default_rng(RNG_SEED)

for fold, (tr_idx, val_idx) in enumerate(cv_splits):
    fold_tr = train_df.iloc[tr_idx].copy()
    fold_tr["_nl_fold"] = fold_tr.groupby("job_id")["relevance_label"].transform(
        lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
    gmean = fold_tr["_nl_fold"].mean()

    for c in HIGH_CARD_COLS:
        smap = fit_te_on_fold(fold_tr, c, gmean)
        val_vals = train_df.iloc[val_idx][c + "_normcat"].map(smap).fillna(gmean).values
        train_df.loc[train_df.index[val_idx], f"{c}_te"] = val_vals
        tr_vals = train_df.iloc[tr_idx][c + "_normcat"].map(smap).fillna(gmean).values
        tr_vals = tr_vals * (1 + rng.normal(0, TE_NOISE, size=len(tr_vals)))
        train_df.loc[train_df.index[tr_idx], f"{c}_te"] = tr_vals
        freq_map = fold_tr[c + "_normcat"].value_counts()
        train_df.loc[train_df.index[tr_idx], f"{c}_freq"] = \
            train_df.iloc[tr_idx][c + "_normcat"].map(freq_map).fillna(0).values
        train_df.loc[train_df.index[val_idx], f"{c}_freq"] = \
            train_df.iloc[val_idx][c + "_normcat"].map(freq_map).fillna(0).values

    X = train_df[final_features]

    tr_order = np.argsort(groups[tr_idx], kind="stable")
    Xtr = X.iloc[tr_idx[tr_order]]
    ytr = y[tr_idx[tr_order]]
    gtr = groups[tr_idx[tr_order]]

    va_order = np.argsort(groups[val_idx], kind="stable")
    Xva = X.iloc[val_idx[va_order]]
    yva = y[val_idx[va_order]]
    gva = groups[val_idx[va_order]]

    _, c_tr = np.unique(gtr, return_counts=True)
    _, c_va = np.unique(gva, return_counts=True)

    lm = lgb.LGBMRanker(**lgb_params)
    lm.fit(Xtr, ytr, group=c_tr,
           eval_set=[(Xva, yva)], eval_group=[c_va],
           callbacks=[lgb.early_stopping(50, verbose=False)])
    lgb_best_iters.append(lm.best_iteration_ or lgb_params['n_estimators'])
    oof_lgb[val_idx] = lm.predict(X.iloc[val_idx])

    cm = CatBoostRanker(**cat_params, early_stopping_rounds=50)
    cm.fit(Pool(Xtr, ytr, group_id=gtr), eval_set=Pool(Xva, yva, group_id=gva))
    cat_best_iters.append(cm.get_best_iteration() or cat_params['iterations'])
    oof_cat[val_idx] = cm.predict(X.iloc[val_idx])

    f_lgb = ndcg_by_group(y[val_idx], oof_lgb[val_idx], groups[val_idx])
    f_cat = ndcg_by_group(y[val_idx], oof_cat[val_idx], groups[val_idx])
    print(f"[fold {fold}] NDCG@10  lgb={f_lgb:.4f}  cat={f_cat:.4f}  "
          f"(lgb_iter={lgb_best_iters[-1]}, cat_iter={cat_best_iters[-1]})")

valid = ~np.isnan(oof_lgb)
r_lgb = group_rank(oof_lgb, groups)
r_cat = group_rank(oof_cat, groups)

print(f"\n[OOF] lgb={ndcg_by_group(y[valid], oof_lgb[valid], groups[valid]):.4f}  "
      f"cat={ndcg_by_group(y[valid], oof_cat[valid], groups[valid]):.4f}")


def power_blend(rl, rc, w, p):
    return w * (rl ** p) + (1 - w) * (rc ** p)


def best_blend(mask):
    bs, bw, bp = -1.0, 0.5, 1.0
    for p in (1.0, 1.5, 2.0):
        for w in np.linspace(0, 1, 41):
            sc = ndcg_by_group(y[mask], power_blend(r_lgb, r_cat, w, p)[mask], groups[mask])
            if sc > bs:
                bs, bw, bp = sc, w, p
    return bw, bp, bs


w_all, p_all, s_all = best_blend(valid)
print(f"[blend] global  w={w_all:.3f} p={p_all}  NDCG@10={s_all:.4f}")


w_seen, p_seen = w_all, p_all
w_cold, p_cold = max(0.0, w_all - 0.15), p_all   


for c in HIGH_CARD_COLS:
    agg = train_df.groupby(c + "_normcat")["_nl_full"].agg(["mean", "count"])
    agg = agg[agg["count"] >= TE_MIN_COUNT]
    smooth = (agg["mean"] * agg["count"] + _global_gmean * TE_K) / (agg["count"] + TE_K)
    train_df[f"{c}_te"] = train_df[c + "_normcat"].map(smooth).fillna(_global_gmean).values
    train_df[f"{c}_freq"] = train_df[c + "_normcat"].map(
        train_df[c + "_normcat"].value_counts()).fillna(0).values

X = train_df[final_features]
_, c_full = np.unique(groups, return_counts=True)
sort_full = np.argsort(groups, kind="stable")

lgb_final_iter = int(np.median(lgb_best_iters))
cat_final_iter = int(np.median(cat_best_iters))
print(f"[final] lgb_iter={lgb_final_iter}  cat_iter={cat_final_iter}")

final_lgb_params = {**lgb_params, 'n_estimators': lgb_final_iter}
final_lgb = lgb.LGBMRanker(**final_lgb_params)
final_lgb.fit(X.iloc[sort_full], y[sort_full], group=c_full)

final_cat_params = {**cat_params, 'iterations': cat_final_iter}
final_cat = CatBoostRanker(**final_cat_params)
final_cat.fit(Pool(X.iloc[sort_full], y[sort_full], group_id=groups[sort_full]))


X_test = test_df[final_features]
test_groups = test_df['job_id'].values

rt_lgb = group_rank(final_lgb.predict(X_test), test_groups)
rt_cat = group_rank(final_cat.predict(X_test), test_groups)

is_cold = ~test_df['job_id'].isin(train_job_ids).values
final_blend = np.empty(len(test_df))
final_blend[~is_cold] = power_blend(rt_lgb, rt_cat, w_seen, p_seen)[~is_cold]
final_blend[is_cold] = power_blend(rt_lgb, rt_cat, w_cold, p_cold)[is_cold]

print(f"[test] cold-start rows: {is_cold.sum()} / {len(is_cold)} "
      f"({100*is_cold.mean():.1f}%)")

sub = pd.DataFrame({'application_id': test_df['application_id'], 'score': final_blend})
sub = sub.sort_values('application_id').reset_index(drop=True)
sub.to_csv('submission_ensemble_final0000000.csv', index=False)
print("[done] wrote submission_ensemble_final.csv")

[pruner] kept 36 static + 8 TE = 44 features
[fold 0] NDCG@10  lgb=0.8927  cat=0.8906  (lgb_iter=830, cat_iter=634)
[fold 1] NDCG@10  lgb=0.8824  cat=0.8869  (lgb_iter=915, cat_iter=1202)
[fold 2] NDCG@10  lgb=0.8823  cat=0.8881  (lgb_iter=841, cat_iter=1089)
[fold 3] NDCG@10  lgb=0.8932  cat=0.8986  (lgb_iter=828, cat_iter=1210)
[fold 4] NDCG@10  lgb=0.8749  cat=0.8799  (lgb_iter=671, cat_iter=791)

[OOF] lgb=0.8851  cat=0.8888
[blend] global  w=0.100 p=1.0  NDCG@10=0.8888
[final] lgb_iter=830  cat_iter=1089
[test] cold-start rows: 48392 / 52700 (91.8%)
[done] wrote submission_ensemble_final.csv


In [1]:
pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 410.9 kB/s  0:05:470:00:0100:10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 265.8/300.2 MB 396.9 kB/s eta 0:01:27
Resuming download nvidia_nccl_cu12-2.30.4-py3-none-manylinux_2_18_x86_64.whl (265.8 MB/300.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 MB 366.8 kB/s  0:01:250:00:0100:22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from collections import Counter
import difflib
import time
import lightgbm as lgb
from catboost import CatBoostRanker, Pool
import xgboost as xgb
from sklearn.metrics import ndcg_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import paired_cosine_distances

RNG_SEED = 42

print("="*60)
print("STARTING THE ULTIMATE 3-WAY ENSEMBLE PIPELINE")
print("="*60)

print("[1/7] Loading datasets...")
train_app = pd.read_csv('applications_train.csv')
test_app = pd.read_csv('applications_test.csv')
jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')

train_df = train_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
test_df = test_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')

train_job_ids = set(train_df['job_id'].unique())

eng_map = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
edu_map = {'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
exchange_rates = {'USD': 1.0, 'EUR': 1.08, 'IRR': 0.00002}

def _truthy(s):
    return s.fillna(0).astype(str).str.lower().isin(['1', 'true', 'yes', 'y', 't'])

print("[2/7] Preprocessing & Engineering Cross-Features...")
_exp_sal_med = None
_sal_min_med = None
_sal_max_med = None

for i, df in enumerate([train_df, test_df]):
    df['missing_data_count'] = df.isnull().sum(axis=1)
    
    df['english_score'] = df['english_proficiency'].map(eng_map).fillna(0)
    df['edu_score'] = df['education_level'].map(edu_map).fillna(0)

    df['application_date'] = pd.to_datetime(df['application_date'], dayfirst=True, format='mixed')
    df['job_posted_date'] = pd.to_datetime(df['job_posted_date'], dayfirst=True, format='mixed')
    df['days_to_apply'] = (df['application_date'] - df['job_posted_date']).dt.days.fillna(0)
    df['apply_month'] = df['application_date'].dt.month.fillna(0)
    df['apply_day_of_week'] = df['application_date'].dt.dayofweek.fillna(0)

    rates = df['salary_currency'].map(exchange_rates).fillna(1.0)
    df['salary_min_usd'] = df['salary_min'] * rates
    df['salary_max_usd'] = df['salary_max'] * rates

    if i == 0: 
        _exp_sal_med = df['expected_salary'].median()
        _sal_min_med = df['salary_min_usd'].median()
        _sal_max_med = df['salary_max_usd'].median()

    df['expected_salary'] = df['expected_salary'].fillna(_exp_sal_med)
    df['salary_min_usd'] = df['salary_min_usd'].fillna(_sal_min_med)
    df['salary_max_usd'] = df['salary_max_usd'].fillna(_sal_max_med)

    df['salary_gap'] = df['expected_salary'] - ((df['salary_min_usd'] + df['salary_max_usd']) / 2)
    df['experience_gap'] = df['years_experience'] - df['min_years_experience']
    
    df['salary_ratio'] = df['expected_salary'] / (df['salary_max_usd'] + 1)
    df['exp_ratio'] = df['years_experience'] / (df['min_years_experience'] + 0.1)

    df['skill_match_ratio'] = df.apply(
        lambda row: len(set(str(row['skills']).split('|')) & set(str(row['required_skills']).split('|'))) /
                    max(1, len(set(str(row['required_skills']).split('|')))), axis=1
    )
    df['title_similarity'] = df.apply(
        lambda row: difflib.SequenceMatcher(
            None, str(row['job_title']).lower(), str(row['current_title']).lower()).ratio(), axis=1
    )
    
    df['title_x_industry'] = df['current_title'].fillna('MISSING').astype(str) + "_" + df['industry'].fillna('MISSING').astype(str)
    df['loc_x_industry'] = df['job_location'].fillna('MISSING').astype(str) + "_" + df['industry'].fillna('MISSING').astype(str)

def tokenize_pipe(s):
    if pd.isna(s): return set()
    return {t.strip().lower() for t in str(s).split("|") if t.strip()}

def tokenize_title(s):
    if pd.isna(s): return set()
    return {w for w in str(s).lower().replace(".", " ").split() if len(w) > 1}

def build_advanced_features(df, fitted_idf=None, fitted_company_freq=None):
    cand_sk = df["skills"].apply(tokenize_pipe)
    req_sk = df["required_skills"].apply(tokenize_pipe)
    certs = df["certifications"].apply(tokenize_pipe)
    job_tok = df["job_title"].apply(tokenize_title)
    cur_tok = df["current_title"].apply(tokenize_title)

    df["skill_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(cand_sk, req_sk)]
    df["missing_required_skills"] = [len(b - a) for a, b in zip(cand_sk, req_sk)]
    df["extra_skills"] = [len(a - b) for a, b in zip(cand_sk, req_sk)]
    df["cert_skill_overlap"] = [len(c & b) for c, b in zip(certs, req_sk)]
    df["cert_title_overlap"] = [len(c & j) for c, j in zip(certs, job_tok)]
    df["n_certifications"] = certs.apply(len)
    df["title_token_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(job_tok, cur_tok)]

    if fitted_idf is None:
        dfreq = Counter()
        for s in req_sk: dfreq.update(s)
        n_docs = len(req_sk)
        idf = {k: np.log((1 + n_docs) / (1 + v)) + 1 for k, v in dfreq.items()}
    else:
        idf = fitted_idf

    vals = []
    for a, b in zip(cand_sk, req_sk):
        if not b:
            vals.append(0.0)
            continue
        num = sum(idf.get(t, 0.0) for t in (a & b))
        den = sum(idf.get(t, 0.0) for t in b)
        vals.append(num / den if den else 0.0)
    df["skill_match_idf"] = vals

    exp = df["years_experience"]
    lo = df["min_years_experience"]
    hi = df["max_years_experience"]
    df["in_experience_band"] = ((exp >= lo) & (exp <= hi)).astype(int)
    df["exp_band_distance"] = np.where(exp < lo, lo - exp, np.where(exp > hi, exp - hi, 0))

    cand_sal = df["expected_salary"]
    jmin = df["salary_min_usd"]
    jmax = df["salary_max_usd"]
    df["salary_band_distance"] = np.where(cand_sal < jmin, jmin - cand_sal, np.where(cand_sal > jmax, cand_sal - jmax, 0))

    prev = df["previous_companies"].apply(tokenize_pipe)
    if fitted_company_freq is None:
        freq = Counter()
        for s in prev: freq.update(s)
    else:
        freq = fitted_company_freq
    df["max_company_freq"] = [max((freq[c] for c in s), default=0) for s in prev]
    df["mean_company_freq"] = [np.mean([freq[c] for c in s]) if s else 0.0 for s in prev]

    df["completeness_x_skill"] = df["profile_completeness"] * df["skill_jaccard"]
    df["idf_x_inband"] = df["skill_match_idf"] * df["in_experience_band"]
    df["idf_x_salband"] = df["skill_match_idf"] * df["salary_band_distance"]

    g = df.groupby('job_id')
    df["expgrank_x_salgrank"] = g["experience_gap"].rank(pct=True) * g["salary_gap"].rank(pct=True)
    df["title_x_expgrank"] = df["title_token_jaccard"] * g["experience_gap"].rank(pct=True)
    df["match_per_salovershoot"] = df["skill_match_idf"] / (df["salary_band_distance"] + 1.0)
    df["match_per_expgap"] = df["skill_match_idf"] / (df["exp_band_distance"] + 1.0)

    rs = g["skill_match_idf"].rank(ascending=False, method="min")
    re = g["exp_band_distance"].rank(ascending=True, method="min")
    df["consensus_rr"] = 1.0 / (rs + 1.0) + 1.0 / (re + 1.0)

    same_loc = (df["candidate_location"].fillna("").astype(str).str.lower() == df["job_location"].fillna("").astype(str).str.lower()).astype(int)
    df["loc_same"] = same_loc

    cols_to_rank = [
        'experience_gap', 'salary_gap', 'skill_match_ratio', 'title_similarity',
        'days_to_apply', 'english_score', 'edu_score', "idf_x_inband", "idf_x_salband",
        "expgrank_x_salgrank", "title_x_expgrank", "match_per_salovershoot", "match_per_expgap", "consensus_rr"
    ]
    for c in cols_to_rank:
        if c in df.columns:
            df[f"{c}_grank"] = g[c].rank(pct=True)
            df[f"{c}_gzscore"] = g[c].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))

    return df, idf, freq

train_df, train_idf, train_company_freq = build_advanced_features(train_df)
test_df, _, _ = build_advanced_features(test_df, fitted_idf=train_idf, fitted_company_freq=train_company_freq)

print("[3/7] Generating SVD Text Features...")
text_cols = [('job_title', 'current_title'), ('required_skills', 'skills')]
for col_job, col_cand in text_cols:
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000)
    corpus_train = train_df[col_job].fillna("") + " " + train_df[col_cand].fillna("")
    tfidf.fit(corpus_train)

    job_tr = tfidf.transform(train_df[col_job].fillna(""))
    cand_tr = tfidf.transform(train_df[col_cand].fillna(""))
    job_te = tfidf.transform(test_df[col_job].fillna(""))
    cand_te = tfidf.transform(test_df[col_cand].fillna(""))

    svd = TruncatedSVD(n_components=10, random_state=RNG_SEED)
    job_tr_s = svd.fit_transform(job_tr)
    cand_tr_s = svd.transform(cand_tr)
    job_te_s = svd.transform(job_te)
    cand_te_s = svd.transform(cand_te)

    train_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_tr_s, cand_tr_s)
    test_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_te_s, cand_te_s)

print("[4/7] Setting up Target Encoding...")
HIGH_CARD_COLS = ['current_title', 'job_location', 'university', 'industry', 'title_x_industry', 'loc_x_industry']
TE_K = 20
TE_NOISE = 0.01

def _norm_cat(series):
    return series.fillna("MISSING").astype(str).str.lower().str.strip()

for c in HIGH_CARD_COLS:
    train_df[c + "_normcat"] = _norm_cat(train_df[c])
    test_df[c + "_normcat"] = _norm_cat(test_df[c])

train_df["_nl_full"] = train_df.groupby("job_id")["relevance_label"].transform(lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
_global_gmean = train_df["_nl_full"].mean()

test_te_maps, test_freq_maps = {}, {}
for c in HIGH_CARD_COLS:
    agg = train_df.groupby(c + "_normcat")["_nl_full"].agg(["mean", "count"])
    smooth = (agg["mean"] * agg["count"] + _global_gmean * TE_K) / (agg["count"] + TE_K)
    test_te_maps[c] = smooth
    test_freq_maps[c] = train_df[c + "_normcat"].value_counts()

def fit_te_on_fold(fold_train_df, cat_col, gmean):
    agg = fold_train_df.groupby(cat_col + "_normcat")["_nl_fold"].agg(["mean", "count"])
    return (agg["mean"] * agg["count"] + gmean * TE_K) / (agg["count"] + TE_K)

cols_to_drop = [
    'application_id', 'candidate_id', 'job_id', 'application_date', 'job_posted_date',
    'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed',
    'company_size', 'industry', 'current_title', 'skills', 'education_level',
    'university', 'previous_companies', 'certifications', 'english_proficiency',
    'candidate_location', 'willing_to_relocate', 'account_created_date', 'relevance_label',
    '_nl_full', '_nl_fold', 'title_x_industry', 'loc_x_industry'
] + [c + "_normcat" for c in HIGH_CARD_COLS]

for c in HIGH_CARD_COLS:
    train_df[f"{c}_te"] = np.nan
    train_df[f"{c}_freq"] = 0.0
    test_df[f"{c}_te"] = test_df[c + "_normcat"].map(test_te_maps[c]).fillna(_global_gmean).values
    test_df[f"{c}_freq"] = test_df[c + "_normcat"].map(test_freq_maps[c]).fillna(0).values

candidate_features = [c for c in train_df.columns if c not in cols_to_drop and train_df[c].dtype in [np.float64, np.float32, np.int64, np.int32]]

print("[5/7] Running Null Importance Pruner (Feature Selection)...")
def null_importance_pruner(X, y, groups, features, n_runs=10):
    sort_idx = np.argsort(groups, kind='stable')
    Xs = X.iloc[sort_idx].reset_index(drop=True)
    ys = y[sort_idx].copy()
    _, counts = np.unique(groups[sort_idx], return_counts=True)

    base = lgb.LGBMRanker(n_estimators=150, learning_rate=0.05, random_state=RNG_SEED, n_jobs=-1, verbosity=-1)
    base.fit(Xs, ys, group=counts)
    actual = base.booster_.feature_importance("gain")

    null = np.zeros((n_runs, len(features)))
    for r in range(n_runs):
        yp = ys.copy()
        start = 0
        rs = np.random.RandomState(r)
        for cnt in counts:
            block = yp[start:start + cnt].copy()
            rs.shuffle(block)
            yp[start:start + cnt] = block
            start += cnt
        m = lgb.LGBMRanker(n_estimators=150, learning_rate=0.05, random_state=r, n_jobs=-1, verbosity=-1)
        m.fit(Xs, yp, group=counts)
        null[r] = m.booster_.feature_importance("gain")

    score = np.log1p(actual) - np.log1p(np.percentile(null, 50, axis=0))
    keep = [f for f, s in zip(features, score) if s > 0]
    return keep if len(keep) >= 15 else features

te_cols = [f"{c}_te" for c in HIGH_CARD_COLS] + [f"{c}_freq" for c in HIGH_CARD_COLS]
static_features = [c for c in candidate_features if c not in te_cols]

kept_static = null_importance_pruner(train_df[static_features], train_df['relevance_label'].values, train_df['job_id'].values, static_features)
final_features = kept_static + te_cols
print(f"      -> Kept {len(kept_static)} static + {len(te_cols)} TE = {len(final_features)} total features.")

def group_rank(pred, groups): 
    return pd.Series(pred).groupby(groups).rank(pct=True).values

def ndcg_by_group(y_true, y_pred, groups, k=10):
    d = pd.DataFrame({"t": y_true, "p": y_pred, "g": groups})
    return float(np.mean([ndcg_score([gr["t"].values], [gr["p"].values], k=k) for _, gr in d.groupby("g") if len(gr) >= 2]))

print("\n[6/7] Training Expanding-Window CV with 3 Models (LGBM, CAT, XGB)...")
train_df = train_df.sort_values('job_id').reset_index(drop=True)
test_df = test_df.sort_values('job_id').reset_index(drop=True)

y = train_df['relevance_label'].values
groups = train_df['job_id'].values

lgb_params = {
    'objective': 'lambdarank', 'metric': 'ndcg', 'n_estimators': 2000,
    'learning_rate': 0.02, 'num_leaves': 63, 'max_depth': 7, 'min_child_samples': 30,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': RNG_SEED, 'n_jobs': -1, 'verbosity': -1
}
cat_params = dict(loss_function="YetiRank", eval_metric="NDCG:top=10", iterations=3500, learning_rate=0.03, depth=6, random_seed=RNG_SEED, verbose=0)
xgb_params = {'objective': 'rank:ndcg', 'eval_metric': 'ndcg@10', 'learning_rate': 0.02, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': RNG_SEED, 'n_estimators': 3000, 'tree_method': 'hist'}

job_dates = train_df.groupby('job_id')['application_date'].max().sort_values()
folds = np.array_split(job_dates.index.to_numpy(), 5)

cv_splits = []
for i in range(1, 5): 
    tr_jobs = np.concatenate(folds[:i])
    val_jobs = folds[i]
    cv_splits.append((train_df.index[train_df['job_id'].isin(tr_jobs)].to_numpy(), train_df.index[train_df['job_id'].isin(val_jobs)].to_numpy()))

oof_lgb, oof_cat, oof_xgb = np.full(len(train_df), np.nan), np.full(len(train_df), np.nan), np.full(len(train_df), np.nan)
lgb_iters, cat_iters, xgb_iters = [], [], []

for fold, (tr_idx, val_idx) in enumerate(cv_splits):
    t0 = time.time()
    print(f"\n--- FOLD {fold+1}/4 ---")
    
    fold_tr = train_df.iloc[tr_idx].copy()
    fold_tr["_nl_fold"] = fold_tr.groupby("job_id")["relevance_label"].transform(lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
    gmean = fold_tr["_nl_fold"].mean()

    for c in HIGH_CARD_COLS:
        smap = fit_te_on_fold(fold_tr, c, gmean)
        train_df.loc[train_df.index[val_idx], f"{c}_te"] = train_df.iloc[val_idx][c + "_normcat"].map(smap).fillna(gmean).values
        train_df.loc[train_df.index[tr_idx], f"{c}_te"] = train_df.iloc[tr_idx][c + "_normcat"].map(smap).fillna(gmean).values
        freq_map = fold_tr[c + "_normcat"].value_counts()
        train_df.loc[train_df.index[tr_idx], f"{c}_freq"] = train_df.iloc[tr_idx][c + "_normcat"].map(freq_map).fillna(0).values
        train_df.loc[train_df.index[val_idx], f"{c}_freq"] = train_df.iloc[val_idx][c + "_normcat"].map(freq_map).fillna(0).values

    X = train_df[final_features]

    tr_order = np.argsort(groups[tr_idx], kind="stable")
    va_order = np.argsort(groups[val_idx], kind="stable")
    
    Xtr, ytr, gtr = X.iloc[tr_idx[tr_order]], y[tr_idx[tr_order]], groups[tr_idx[tr_order]]
    Xva, yva, gva = X.iloc[val_idx[va_order]], y[val_idx[va_order]], groups[val_idx[va_order]]
    _, c_tr = np.unique(gtr, return_counts=True)
    _, c_va = np.unique(gva, return_counts=True)

    lm = lgb.LGBMRanker(**lgb_params)
    lm.fit(Xtr, ytr, group=c_tr, eval_set=[(Xva, yva)], eval_group=[c_va], callbacks=[lgb.early_stopping(50, verbose=False)])
    lgb_iters.append(lm.best_iteration_ or lgb_params['n_estimators'])
    oof_lgb[val_idx] = lm.predict(X.iloc[val_idx])

    cm = CatBoostRanker(**cat_params, early_stopping_rounds=50)
    cm.fit(Pool(Xtr, ytr, group_id=gtr), eval_set=Pool(Xva, yva, group_id=gva))
    cat_iters.append(cm.get_best_iteration() or cat_params['iterations'])
    oof_cat[val_idx] = cm.predict(X.iloc[val_idx])

    xm = xgb.XGBRanker(**xgb_params, early_stopping_rounds=50)
    xm.fit(Xtr, ytr, qid=gtr, eval_set=[(Xva, yva)], eval_qid=[gva], verbose=False)
    xgb_iters.append(xm.best_iteration)
    oof_xgb[val_idx] = xm.predict(X.iloc[val_idx])
    
    f_lgb = ndcg_by_group(y[val_idx], oof_lgb[val_idx], groups[val_idx])
    f_cat = ndcg_by_group(y[val_idx], oof_cat[val_idx], groups[val_idx])
    f_xgb = ndcg_by_group(y[val_idx], oof_xgb[val_idx], groups[val_idx])
    
    print(f"  Time: {time.time()-t0:.1f}s")
    print(f"  LGBM -> NDCG: {f_lgb:.5f} | Best Iter: {lgb_iters[-1]}")
    print(f"  CAT  -> NDCG: {f_cat:.5f} | Best Iter: {cat_iters[-1]}")
    print(f"  XGB  -> NDCG: {f_xgb:.5f} | Best Iter: {xgb_iters[-1]}")

valid = ~np.isnan(oof_lgb)
r_lgb, r_cat, r_xgb = group_rank(oof_lgb, groups), group_rank(oof_cat, groups), group_rank(oof_xgb, groups)

print("\n" + "="*60)
print("[7/7] Optimal Blend Search & Final Prediction...")
print(f"Average Optimal Iters -> LGBM: {int(np.median(lgb_iters))}, CAT: {int(np.median(cat_iters))}, XGB: {int(np.median(xgb_iters))}")

bs, best_w = -1.0, (0.34, 0.33, 0.33)
for w_lgb in np.linspace(0, 1, 51):
    for w_cat in np.linspace(0, 1 - w_lgb, 51):
        w_xgb = max(0.0, 1.0 - w_lgb - w_cat)
        combo = w_lgb * r_lgb + w_cat * r_cat + w_xgb * r_xgb
        sc = ndcg_by_group(y[valid], combo[valid], groups[valid])
        if sc > bs: bs, best_w = sc, (w_lgb, w_cat, w_xgb)

print(f"*** BEST BLEND WEIGHTS ***")
print(f"LGB={best_w[0]:.2f}, CAT={best_w[1]:.2f}, XGB={best_w[2]:.2f} ---> OOF NDCG: {bs:.6f}")
print("="*60)

for c in HIGH_CARD_COLS:
    train_df[f"{c}_te"] = train_df[c + "_normcat"].map(test_te_maps[c]).fillna(_global_gmean).values
    train_df[f"{c}_freq"] = train_df[c + "_normcat"].map(test_freq_maps[c]).fillna(0).values

X = train_df[final_features]
_, c_full = np.unique(groups, return_counts=True)
sort_full = np.argsort(groups, kind="stable")
X_sorted, y_sorted, groups_sorted = X.iloc[sort_full], y[sort_full], groups[sort_full]

print("Training Final Models on 100% of Data...")
final_lgb = lgb.LGBMRanker(**{**lgb_params, 'n_estimators': int(np.median(lgb_iters))})
final_lgb.fit(X_sorted, y_sorted, group=c_full)

final_cat = CatBoostRanker(**{**cat_params, 'iterations': int(np.median(cat_iters))})
final_cat.fit(Pool(X_sorted, y_sorted, group_id=groups_sorted))

final_xgb = xgb.XGBRanker(**{**xgb_params, 'n_estimators': int(np.median(xgb_iters))})
final_xgb.fit(X_sorted, y_sorted, qid=groups_sorted)

X_test = test_df[final_features]
t_groups = test_df['job_id'].values

pred_lgb = group_rank(final_lgb.predict(X_test), t_groups)
pred_cat = group_rank(final_cat.predict(X_test), t_groups)
pred_xgb = group_rank(final_xgb.predict(X_test), t_groups)

final_blend = best_w[0] * pred_lgb + best_w[1] * pred_cat + best_w[2] * pred_xgb

sub = pd.DataFrame({'application_id': test_df['application_id'], 'score': final_blend})
sub.to_csv('submission_ultimate_3way11.csv', index=False)
print("Done! 'submission_ultimate_3way.csv' is ready.")

STARTING THE ULTIMATE 3-WAY ENSEMBLE PIPELINE
[1/7] Loading datasets...
[2/7] Preprocessing & Engineering Cross-Features...
[3/7] Generating SVD Text Features...
[4/7] Setting up Target Encoding...
[5/7] Running Null Importance Pruner (Feature Selection)...
      -> Kept 35 static + 12 TE = 47 total features.

[6/7] Training Expanding-Window CV with 3 Models (LGBM, CAT, XGB)...

--- FOLD 1/4 ---
  Time: 42.1s
  LGBM -> NDCG: 0.83118 | Best Iter: 96
  CAT  -> NDCG: 0.86436 | Best Iter: 822
  XGB  -> NDCG: 0.85379 | Best Iter: 1017

--- FOLD 2/4 ---
  Time: 99.6s
  LGBM -> NDCG: 0.86613 | Best Iter: 298
  CAT  -> NDCG: 0.87836 | Best Iter: 521
  XGB  -> NDCG: 0.87082 | Best Iter: 995

--- FOLD 3/4 ---
  Time: 95.1s
  LGBM -> NDCG: 0.88562 | Best Iter: 601
  CAT  -> NDCG: 0.89235 | Best Iter: 616
  XGB  -> NDCG: 0.88151 | Best Iter: 897

--- FOLD 4/4 ---
  Time: 135.1s
  LGBM -> NDCG: 0.86362 | Best Iter: 284
  CAT  -> NDCG: 0.87718 | Best Iter: 703
  XGB  -> NDCG: 0.87209 | Best Iter: 12

In [4]:
import numpy as np
import pandas as pd
from collections import Counter
import difflib
import time

import lightgbm as lgb
from catboost import CatBoostRanker, Pool
import xgboost as xgb
from sklearn.metrics import ndcg_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import paired_cosine_distances

RNG_SEED = 42

print("=" * 60)
print("3-WAY ENSEMBLE PIPELINE (LGBM + CatBoost + XGBoost)")
print("=" * 60)

# ----------------------------------------------------------------------------
# 1. Load Data
# ----------------------------------------------------------------------------
print("[1/7] Loading datasets...")
train_app = pd.read_csv('applications_train.csv')
test_app = pd.read_csv('applications_test.csv')
jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')

train_df = train_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
test_df = test_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')

print(f"      -> Train shape: {train_df.shape}, Test shape: {test_df.shape}")
train_job_ids = set(train_df['job_id'].unique())
cold_frac = (~test_df['job_id'].isin(train_job_ids)).mean()
print(f"      -> Cold-start fraction in test: {cold_frac:.3f}")

eng_map = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
edu_map = {'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
exchange_rates = {'USD': 1.0, 'EUR': 1.08, 'IRR': 0.00002}


def _truthy(s):
    return s.fillna(0).astype(str).str.lower().isin(['1', 'true', 'yes', 'y', 't'])


# ----------------------------------------------------------------------------
# 2. Base Preprocessing & Cross-Features
#   Imputation medians captured on TRAIN only, reused on test.
# ----------------------------------------------------------------------------
print("[2/7] Preprocessing & engineering cross-features...")
_exp_sal_med = None
_sal_min_med = None
_sal_max_med = None

for i, df in enumerate([train_df, test_df]):
    df['english_score'] = df['english_proficiency'].map(eng_map).fillna(0)
    df['edu_score'] = df['education_level'].map(edu_map).fillna(0)

    df['application_date'] = pd.to_datetime(df['application_date'], dayfirst=True, format='mixed')
    df['job_posted_date'] = pd.to_datetime(df['job_posted_date'], dayfirst=True, format='mixed')
    df['days_to_apply'] = (df['application_date'] - df['job_posted_date']).dt.days.fillna(0)

    rates = df['salary_currency'].map(exchange_rates).fillna(1.0)
    df['salary_min_usd'] = df['salary_min'] * rates
    df['salary_max_usd'] = df['salary_max'] * rates

    if i == 0:
        _exp_sal_med = df['expected_salary'].median()
        _sal_min_med = df['salary_min_usd'].median()
        _sal_max_med = df['salary_max_usd'].median()

    df['expected_salary'] = df['expected_salary'].fillna(_exp_sal_med)
    df['salary_min_usd'] = df['salary_min_usd'].fillna(_sal_min_med)
    df['salary_max_usd'] = df['salary_max_usd'].fillna(_sal_max_med)

    df['salary_gap'] = df['expected_salary'] - ((df['salary_min_usd'] + df['salary_max_usd']) / 2)
    df['experience_gap'] = df['years_experience'] - df['min_years_experience']

    df['skill_match_ratio'] = df.apply(
        lambda row: len(set(str(row['skills']).split('|')) & set(str(row['required_skills']).split('|'))) /
                    max(1, len(set(str(row['required_skills']).split('|')))), axis=1
    )
    df['title_similarity'] = df.apply(
        lambda row: difflib.SequenceMatcher(
            None, str(row['job_title']).lower(), str(row['current_title']).lower()).ratio(), axis=1
    )

    df['title_x_industry'] = (df['current_title'].fillna('MISSING').astype(str) + "_" +
                              df['industry'].fillna('MISSING').astype(str))
    df['loc_x_industry'] = (df['job_location'].fillna('MISSING').astype(str) + "_" +
                            df['industry'].fillna('MISSING').astype(str))


def tokenize_pipe(s):
    if pd.isna(s):
        return set()
    return {t.strip().lower() for t in str(s).split("|") if t.strip()}


def tokenize_title(s):
    if pd.isna(s):
        return set()
    return {w for w in str(s).lower().replace(".", " ").split() if len(w) > 1}


def build_advanced_features(df, fitted_idf=None, fitted_company_freq=None):
    cand_sk = df["skills"].apply(tokenize_pipe)
    req_sk = df["required_skills"].apply(tokenize_pipe)
    certs = df["certifications"].apply(tokenize_pipe)
    job_tok = df["job_title"].apply(tokenize_title)
    cur_tok = df["current_title"].apply(tokenize_title)

    df["skill_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(cand_sk, req_sk)]
    df["missing_required_skills"] = [len(b - a) for a, b in zip(cand_sk, req_sk)]
    df["extra_skills"] = [len(a - b) for a, b in zip(cand_sk, req_sk)]
    df["cert_skill_overlap"] = [len(c & b) for c, b in zip(certs, req_sk)]
    df["cert_title_overlap"] = [len(c & j) for c, j in zip(certs, job_tok)]
    df["n_certifications"] = certs.apply(len)
    df["title_token_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(job_tok, cur_tok)]

    if fitted_idf is None:
        dfreq = Counter()
        for s in req_sk:
            dfreq.update(s)
        n_docs = len(req_sk)
        idf = {k: np.log((1 + n_docs) / (1 + v)) + 1 for k, v in dfreq.items()}
    else:
        idf = fitted_idf

    vals = []
    for a, b in zip(cand_sk, req_sk):
        if not b:
            vals.append(0.0)
            continue
        num = sum(idf.get(t, 0.0) for t in (a & b))
        den = sum(idf.get(t, 0.0) for t in b)
        vals.append(num / den if den else 0.0)
    df["skill_match_idf"] = vals

    exp = df["years_experience"]
    lo = df["min_years_experience"]
    hi = df["max_years_experience"]
    df["in_experience_band"] = ((exp >= lo) & (exp <= hi)).astype(int)
    df["exp_band_distance"] = np.where(exp < lo, lo - exp, np.where(exp > hi, exp - hi, 0))

    cand_sal = df["expected_salary"]
    jmin = df["salary_min_usd"]
    jmax = df["salary_max_usd"]
    df["salary_band_distance"] = np.where(cand_sal < jmin, jmin - cand_sal,
                                          np.where(cand_sal > jmax, cand_sal - jmax, 0))

    prev = df["previous_companies"].apply(tokenize_pipe)
    if fitted_company_freq is None:
        freq = Counter()
        for s in prev:
            freq.update(s)
    else:
        freq = fitted_company_freq
    df["max_company_freq"] = [max((freq[c] for c in s), default=0) for s in prev]
    df["mean_company_freq"] = [np.mean([freq[c] for c in s]) if s else 0.0 for s in prev]

    df["completeness_x_skill"] = df["profile_completeness"] * df["skill_jaccard"]
    df["idf_x_inband"] = df["skill_match_idf"] * df["in_experience_band"]
    df["idf_x_salband"] = df["skill_match_idf"] * df["salary_band_distance"]

    g = df.groupby('job_id')
    df["expgrank_x_salgrank"] = g["experience_gap"].rank(pct=True) * g["salary_gap"].rank(pct=True)
    df["title_x_expgrank"] = df["title_token_jaccard"] * g["experience_gap"].rank(pct=True)
    df["match_per_salovershoot"] = df["skill_match_idf"] / (df["salary_band_distance"] + 1.0)
    df["match_per_expgap"] = df["skill_match_idf"] / (df["exp_band_distance"] + 1.0)

    # ---- MAGIC #1: reciprocal-rank consensus of the two strongest matchers ----
    rs = g["skill_match_idf"].rank(ascending=False, method="min")
    re = g["exp_band_distance"].rank(ascending=True, method="min")
    df["consensus_rr"] = 1.0 / (rs + 1.0) + 1.0 / (re + 1.0)

    # ---- MAGIC #2: competitive standout vs the best candidate in the group ----
    df["skill_gap_to_best"] = g["skill_match_idf"].transform("max") - df["skill_match_idf"]
    df["skill_z"] = g["skill_match_idf"].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))

    # ---- MAGIC #3: location compatibility / friction (uses dropped columns) ----
    same_loc = (df["candidate_location"].fillna("").astype(str).str.lower() ==
                df["job_location"].fillna("").astype(str).str.lower()).astype(int)
    remote_ok = _truthy(df["remote_allowed"]) if "remote_allowed" in df.columns else pd.Series(False, index=df.index)
    df["loc_same"] = same_loc
    df["loc_friction"] = ((1 - same_loc) * (~remote_ok).astype(int)).astype(int)

    cols_to_rank = [
        'experience_gap', 'salary_gap', 'skill_match_ratio', 'title_similarity',
        'days_to_apply', 'english_score', 'edu_score', "idf_x_inband", "idf_x_salband",
        "expgrank_x_salgrank", "title_x_expgrank", "match_per_salovershoot", "match_per_expgap",
        "consensus_rr", "skill_gap_to_best",
    ]
    for c in cols_to_rank:
        if c in df.columns:
            df[f"{c}_grank"] = g[c].rank(pct=True)
            df[f"{c}_gzscore"] = g[c].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))

    return df, idf, freq


train_df, train_idf, train_company_freq = build_advanced_features(train_df)
test_df, _, _ = build_advanced_features(test_df, fitted_idf=train_idf, fitted_company_freq=train_company_freq)

# ----------------------------------------------------------------------------
# 3. LSA / SVD text features (similarity only; raw components excluded)
# ----------------------------------------------------------------------------
print("[3/7] Generating SVD text features...")
text_cols = [('job_title', 'current_title'), ('required_skills', 'skills')]
for col_job, col_cand in text_cols:
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000)
    corpus_train = train_df[col_job].fillna("") + " " + train_df[col_cand].fillna("")
    tfidf.fit(corpus_train)

    job_tr = tfidf.transform(train_df[col_job].fillna(""))
    cand_tr = tfidf.transform(train_df[col_cand].fillna(""))
    job_te = tfidf.transform(test_df[col_job].fillna(""))
    cand_te = tfidf.transform(test_df[col_cand].fillna(""))

    svd = TruncatedSVD(n_components=10, random_state=RNG_SEED)
    job_tr_s = svd.fit_transform(job_tr)
    cand_tr_s = svd.transform(cand_tr)
    job_te_s = svd.transform(job_te)
    cand_te_s = svd.transform(cand_te)

    train_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_tr_s, cand_tr_s)
    test_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_te_s, cand_te_s)

    # sign-invariant dot summaries (safe on unseen jobs)
    prod_tr = job_tr_s * cand_tr_s
    prod_te = job_te_s * cand_te_s
    train_df[f'{col_job}_svd_dotmax'] = prod_tr.max(axis=1)
    test_df[f'{col_job}_svd_dotmax'] = prod_te.max(axis=1)

# ----------------------------------------------------------------------------
# 4. Target Encoding setup
# ----------------------------------------------------------------------------
print("[4/7] Setting up target encoding...")
HIGH_CARD_COLS = ['current_title', 'job_location', 'university', 'industry',
                  'title_x_industry', 'loc_x_industry']
TE_K = 20
TE_NOISE = 0.01


def _norm_cat(series):
    return series.fillna("MISSING").astype(str).str.lower().str.strip()


for c in HIGH_CARD_COLS:
    train_df[c + "_normcat"] = _norm_cat(train_df[c])
    test_df[c + "_normcat"] = _norm_cat(test_df[c])

train_df["_nl_full"] = train_df.groupby("job_id")["relevance_label"].transform(
    lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
_global_gmean = train_df["_nl_full"].mean()

test_te_maps, test_freq_maps = {}, {}
for c in HIGH_CARD_COLS:
    agg = train_df.groupby(c + "_normcat")["_nl_full"].agg(["mean", "count"])
    smooth = (agg["mean"] * agg["count"] + _global_gmean * TE_K) / (agg["count"] + TE_K)
    test_te_maps[c] = smooth
    test_freq_maps[c] = train_df[c + "_normcat"].value_counts()


def fit_te_on_fold(fold_train_df, cat_col, gmean):
    agg = fold_train_df.groupby(cat_col + "_normcat")["_nl_fold"].agg(["mean", "count"])
    return (agg["mean"] * agg["count"] + gmean * TE_K) / (agg["count"] + TE_K)


cols_to_drop = [
    'application_id', 'candidate_id', 'job_id', 'application_date', 'job_posted_date',
    'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed',
    'company_size', 'industry', 'current_title', 'skills', 'education_level',
    'university', 'previous_companies', 'certifications', 'english_proficiency',
    'candidate_location', 'willing_to_relocate', 'account_created_date', 'relevance_label',
    '_nl_full', '_nl_fold', 'title_x_industry', 'loc_x_industry',
] + [c + "_normcat" for c in HIGH_CARD_COLS]

for c in HIGH_CARD_COLS:
    train_df[f"{c}_te"] = np.nan
    train_df[f"{c}_freq"] = 0.0
    test_df[f"{c}_te"] = test_df[c + "_normcat"].map(test_te_maps[c]).fillna(_global_gmean).values
    test_df[f"{c}_freq"] = test_df[c + "_normcat"].map(test_freq_maps[c]).fillna(0).values

candidate_features = [c for c in train_df.columns
                      if c not in cols_to_drop and train_df[c].dtype in
                      [np.float64, np.float32, np.int64, np.int32]]

# ----------------------------------------------------------------------------
# 5. Null Importance Pruner
# ----------------------------------------------------------------------------
print("[5/7] Running null-importance pruner...")


def null_importance_pruner(X, y, groups, features, n_runs=10):
    sort_idx = np.argsort(groups, kind='stable')
    Xs = X.iloc[sort_idx].reset_index(drop=True)
    ys = y[sort_idx].copy()
    _, counts = np.unique(groups[sort_idx], return_counts=True)

    base = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, num_leaves=63,
                          random_state=RNG_SEED, n_jobs=-1, verbosity=-1)
    base.fit(Xs, ys, group=counts)
    actual = base.booster_.feature_importance("gain")

    null = np.zeros((n_runs, len(features)))
    for r in range(n_runs):
        yp = ys.copy()
        start = 0
        rs = np.random.RandomState(1000 + r)
        for cnt in counts:
            block = yp[start:start + cnt].copy()
            rs.shuffle(block)
            yp[start:start + cnt] = block
            start += cnt
        m = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, num_leaves=63,
                           random_state=r, n_jobs=-1, verbosity=-1)
        m.fit(Xs, yp, group=counts)
        null[r] = m.booster_.feature_importance("gain")

    score = np.log1p(actual) - np.log1p(np.percentile(null, 75, axis=0))
    keep = [f for f, s in zip(features, score) if s > 0]
    if len(keep) < 15:
        order = np.argsort(actual)[::-1]
        keep = [features[i] for i in order[:50]]
    return keep


te_cols = [f"{c}_te" for c in HIGH_CARD_COLS] + [f"{c}_freq" for c in HIGH_CARD_COLS]
static_features = [c for c in candidate_features if c not in te_cols]

kept_static = null_importance_pruner(train_df[static_features],
                                     train_df['relevance_label'].values,
                                     train_df['job_id'].values, static_features)
final_features = kept_static + te_cols
print(f"      -> Kept {len(kept_static)} static + {len(te_cols)} TE = {len(final_features)} total features.")


# ----------------------------------------------------------------------------
# Metric helpers
# ----------------------------------------------------------------------------
def group_rank(pred, groups):
    return pd.Series(pred).groupby(groups).rank(pct=True).values


def ndcg_by_group(y_true, y_pred, groups, k=10):
    d = pd.DataFrame({"t": y_true, "p": y_pred, "g": groups})
    return float(np.mean([ndcg_score([gr["t"].values], [gr["p"].values], k=k)
                          for _, gr in d.groupby("g") if len(gr) >= 2]))


# ----------------------------------------------------------------------------
# 6. Symmetric 5-fold job CV with 3 models
# ----------------------------------------------------------------------------
print("\n[6/7] Training symmetric 5-fold CV (LGBM + CatBoost + XGBoost)...")
train_df = train_df.sort_values('job_id').reset_index(drop=True)
test_df = test_df.sort_values('job_id').reset_index(drop=True)

y = train_df['relevance_label'].values
groups = train_df['job_id'].values

lgb_params = {
    'objective': 'lambdarank', 'metric': 'ndcg', 'eval_at': [10],
    'lambdarank_truncation_level': 12,
    'label_gain': [0, 1, 3, 7, 15],
    'n_estimators': 3000, 'learning_rate': 0.02,
    'num_leaves': 63, 'max_depth': 7,
    'min_child_samples': 30, 'min_split_gain': 0.0,
    'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8,
    'reg_alpha': 0.5, 'reg_lambda': 5.0,
    'random_state': RNG_SEED, 'n_jobs': -1, 'verbosity': -1,
}

cat_params = dict(
    loss_function="YetiRank", eval_metric="NDCG:top=10",
    iterations=3500, learning_rate=0.03, depth=6, l2_leaf_reg=5,
    random_seed=RNG_SEED, verbose=0,
)

xgb_params = {
    'objective': 'rank:ndcg', 'eval_metric': 'ndcg@10',
    'lambdarank_pair_method': 'topk', 'lambdarank_num_pair_per_sample': 12,
    'ndcg_exp_gain': True,
    'learning_rate': 0.02, 'max_depth': 6, 'min_child_weight': 5,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 5.0,
    'n_estimators': 2000, 'tree_method': 'hist', 'random_state': RNG_SEED,
    'early_stopping_rounds': 50,  # in constructor for xgboost>=2.1 / 3.x
}

job_dates = train_df.groupby('job_id')['application_date'].max().sort_values()
folds = np.array_split(job_dates.index.to_numpy(), 5)

cv_splits = []
for i in range(5):  # symmetric: every fold trains on the other 4 (full data)
    val_jobs = folds[i]
    tr_jobs = np.concatenate([folds[j] for j in range(5) if j != i])
    cv_splits.append((train_df.index[train_df['job_id'].isin(tr_jobs)].to_numpy(),
                      train_df.index[train_df['job_id'].isin(val_jobs)].to_numpy()))

oof_lgb = np.full(len(train_df), np.nan)
oof_cat = np.full(len(train_df), np.nan)
oof_xgb = np.full(len(train_df), np.nan)
lgb_iters, cat_iters, xgb_iters = [], [], []
rng = np.random.default_rng(RNG_SEED)

for fold, (tr_idx, val_idx) in enumerate(cv_splits):
    t0 = time.time()
    print(f"\n--- FOLD {fold + 1}/5 ---")

    # in-fold target encoding (leak-free wrt this split)
    fold_tr = train_df.iloc[tr_idx].copy()
    fold_tr["_nl_fold"] = fold_tr.groupby("job_id")["relevance_label"].transform(
        lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
    gmean = fold_tr["_nl_fold"].mean()

    for c in HIGH_CARD_COLS:
        smap = fit_te_on_fold(fold_tr, c, gmean)
        val_vals = train_df.iloc[val_idx][c + "_normcat"].map(smap).fillna(gmean).values
        train_df.loc[train_df.index[val_idx], f"{c}_te"] = val_vals
        tr_vals = train_df.iloc[tr_idx][c + "_normcat"].map(smap).fillna(gmean).values
        tr_vals = tr_vals * (1 + rng.normal(0, TE_NOISE, size=len(tr_vals)))
        train_df.loc[train_df.index[tr_idx], f"{c}_te"] = tr_vals
        freq_map = fold_tr[c + "_normcat"].value_counts()
        train_df.loc[train_df.index[tr_idx], f"{c}_freq"] = \
            train_df.iloc[tr_idx][c + "_normcat"].map(freq_map).fillna(0).values
        train_df.loc[train_df.index[val_idx], f"{c}_freq"] = \
            train_df.iloc[val_idx][c + "_normcat"].map(freq_map).fillna(0).values

    X = train_df[final_features]

    tr_order = np.argsort(groups[tr_idx], kind="stable")
    va_order = np.argsort(groups[val_idx], kind="stable")
    Xtr, ytr, gtr = X.iloc[tr_idx[tr_order]], y[tr_idx[tr_order]], groups[tr_idx[tr_order]]
    Xva, yva, gva = X.iloc[val_idx[va_order]], y[val_idx[va_order]], groups[val_idx[va_order]]
    _, c_tr = np.unique(gtr, return_counts=True)
    _, c_va = np.unique(gva, return_counts=True)

    # 1. LightGBM
    lm = lgb.LGBMRanker(**lgb_params)
    lm.fit(Xtr, ytr, group=c_tr, eval_set=[(Xva, yva)], eval_group=[c_va],
           callbacks=[lgb.early_stopping(50, verbose=False)])
    lgb_iters.append(lm.best_iteration_ or lgb_params['n_estimators'])
    oof_lgb[val_idx] = lm.predict(X.iloc[val_idx])

    # 2. CatBoost
    cm = CatBoostRanker(**cat_params, early_stopping_rounds=50)
    cm.fit(Pool(Xtr, ytr, group_id=gtr), eval_set=Pool(Xva, yva, group_id=gva))
    cat_iters.append(cm.get_best_iteration() or cat_params['iterations'])
    oof_cat[val_idx] = cm.predict(X.iloc[val_idx])

    # 3. XGBoost
    xm = xgb.XGBRanker(**xgb_params)
    xm.fit(Xtr, ytr, qid=gtr, eval_set=[(Xva, yva)], eval_qid=[gva], verbose=False)
    xgb_iters.append(xm.best_iteration)
    oof_xgb[val_idx] = xm.predict(X.iloc[val_idx])

    f_lgb = ndcg_by_group(y[val_idx], oof_lgb[val_idx], groups[val_idx])
    f_cat = ndcg_by_group(y[val_idx], oof_cat[val_idx], groups[val_idx])
    f_xgb = ndcg_by_group(y[val_idx], oof_xgb[val_idx], groups[val_idx])

    print(f"  Time: {time.time() - t0:.1f}s")
    print(f"  LGBM -> NDCG: {f_lgb:.5f} | Best Iter: {lgb_iters[-1]}")
    print(f"  CAT  -> NDCG: {f_cat:.5f} | Best Iter: {cat_iters[-1]}")
    print(f"  XGB  -> NDCG: {f_xgb:.5f} | Best Iter: {xgb_iters[-1]}")

valid = ~np.isnan(oof_lgb)
r_lgb = group_rank(oof_lgb, groups)
r_cat = group_rank(oof_cat, groups)
r_xgb = group_rank(oof_xgb, groups)

print("\n[OOF per-model]")
print(f"  LGBM: {ndcg_by_group(y[valid], oof_lgb[valid], groups[valid]):.5f}")
print(f"  CAT : {ndcg_by_group(y[valid], oof_cat[valid], groups[valid]):.5f}")
print(f"  XGB : {ndcg_by_group(y[valid], oof_xgb[valid], groups[valid]):.5f}")

# ----------------------------------------------------------------------------
# 7. Blend search (with shrink toward equal weights) + final prediction
# ----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("[7/7] Blend search & final prediction...")
print(f"Median iters -> LGBM: {int(np.median(lgb_iters))}, "
      f"CAT: {int(np.median(cat_iters))}, XGB: {int(np.median(xgb_iters))}")

bs, raw_w = -1.0, (1 / 3, 1 / 3, 1 / 3)
for w_lgb in np.linspace(0, 1, 21):
    for w_cat in np.linspace(0, 1 - w_lgb, 21):
        w_xgb = max(0.0, 1.0 - w_lgb - w_cat)
        combo = w_lgb * r_lgb + w_cat * r_cat + w_xgb * r_xgb
        sc = ndcg_by_group(y[valid], combo[valid], groups[valid])
        if sc > bs:
            bs, raw_w = sc, (w_lgb, w_cat, w_xgb)

# shrink 25% toward equal weights — guards against OOF-overfit blends
eq = np.array([1 / 3, 1 / 3, 1 / 3])
best_w = tuple(0.75 * np.array(raw_w) + 0.25 * eq)
shrunk_score = ndcg_by_group(
    y[valid], (best_w[0] * r_lgb + best_w[1] * r_cat + best_w[2] * r_xgb)[valid], groups[valid])

print(f"Raw best   -> LGB={raw_w[0]:.2f} CAT={raw_w[1]:.2f} XGB={raw_w[2]:.2f}  OOF={bs:.5f}")
print(f"Shrunk used-> LGB={best_w[0]:.2f} CAT={best_w[1]:.2f} XGB={best_w[2]:.2f}  OOF={shrunk_score:.5f}")
print("=" * 60)

# global TE for final full-data fit (matches what test will see)
for c in HIGH_CARD_COLS:
    train_df[f"{c}_te"] = train_df[c + "_normcat"].map(test_te_maps[c]).fillna(_global_gmean).values
    train_df[f"{c}_freq"] = train_df[c + "_normcat"].map(test_freq_maps[c]).fillna(0).values

X = train_df[final_features]
_, c_full = np.unique(groups, return_counts=True)
sort_full = np.argsort(groups, kind="stable")
X_sorted, y_sorted, groups_sorted = X.iloc[sort_full], y[sort_full], groups[sort_full]

print("Training final models on 100% of data...")
final_lgb = lgb.LGBMRanker(**{**lgb_params, 'n_estimators': int(np.median(lgb_iters))})
final_lgb.fit(X_sorted, y_sorted, group=c_full)

final_cat = CatBoostRanker(**{**cat_params, 'iterations': int(np.median(cat_iters))})
final_cat.fit(Pool(X_sorted, y_sorted, group_id=groups_sorted))

final_xgb_params = {**xgb_params, 'n_estimators': int(np.median(xgb_iters))}
final_xgb_params.pop('early_stopping_rounds', None)  # no eval set on full-data fit
final_xgb = xgb.XGBRanker(**final_xgb_params)
final_xgb.fit(X_sorted, y_sorted, qid=groups_sorted)

X_test = test_df[final_features]
t_groups = test_df['job_id'].values

pred_lgb = group_rank(final_lgb.predict(X_test), t_groups)
pred_cat = group_rank(final_cat.predict(X_test), t_groups)
pred_xgb = group_rank(final_xgb.predict(X_test), t_groups)

final_blend = best_w[0] * pred_lgb + best_w[1] * pred_cat + best_w[2] * pred_xgb

sub = pd.DataFrame({'application_id': test_df['application_id'], 'score': final_blend})
sub = sub.sort_values('application_id').reset_index(drop=True)
sub.to_csv('submission_ultimate_3way64654.csv', index=False)
print("Done. Wrote 'submission_ultimate_3way.csv'.")

3-WAY ENSEMBLE PIPELINE (LGBM + CatBoost + XGBoost)
[1/7] Loading datasets...
      -> Train shape: (118772, 30), Test shape: (52700, 29)
      -> Cold-start fraction in test: 0.918
[2/7] Preprocessing & engineering cross-features...
[3/7] Generating SVD text features...
[4/7] Setting up target encoding...
[5/7] Running null-importance pruner...
      -> Kept 30 static + 12 TE = 42 total features.

[6/7] Training symmetric 5-fold CV (LGBM + CatBoost + XGBoost)...

--- FOLD 1/5 ---
  Time: 176.8s
  LGBM -> NDCG: 0.88480 | Best Iter: 698
  CAT  -> NDCG: 0.88456 | Best Iter: 689
  XGB  -> NDCG: 0.88597 | Best Iter: 1676

--- FOLD 2/5 ---
  Time: 213.1s
  LGBM -> NDCG: 0.87504 | Best Iter: 745
  CAT  -> NDCG: 0.88276 | Best Iter: 1425
  XGB  -> NDCG: 0.87387 | Best Iter: 1210

--- FOLD 3/5 ---
  Time: 155.9s
  LGBM -> NDCG: 0.87446 | Best Iter: 486
  CAT  -> NDCG: 0.88489 | Best Iter: 1128
  XGB  -> NDCG: 0.87190 | Best Iter: 939

--- FOLD 4/5 ---
  Time: 194.1s
  LGBM -> NDCG: 0.88688 | B